# Task 3 — Jointly Trained Soft Mixture-of-Experts Restoration

$$w=\mathrm{softmax}\!\left(\frac{G(\tilde x)}{\tau}\right),\qquad \hat x = w_0\tilde x + w_1A_{\text{salt}}(\tilde x)+w_2A_{\text{blur}}(\tilde x)+w_3A_{\text{occ}}(\tilde x)$$
$$\mathcal{L}_{MoE}=\lambda_1\mathcal{L}_{L1}+\lambda_s(1-\mathrm{SSIM})+\lambda_c\mathcal{L}_{CE}+\lambda_b\mathcal{L}_{balance},\qquad \mathcal{L}_{balance}=\sum_{k=0}^{3}\left(\bar w_k-\tfrac14\right)^2$$

* **Initialisation (no random components):** the gate $G$ is the trained Task 2 classifier, the experts are the three trained Task 2 specialists, and branch 0 is the identity.
* **Schedule:** a short warm-up with the experts **frozen** (only the gate trains), then **joint fine-tuning** of gate and experts with a smaller learning rate.
* **Balanced batches** (B/4 of each condition), so $\bar w_k$ is measured on the distribution the balance term assumes.
* **Limiting case:** as $\tau\to0$ the mixture becomes exactly the Task 2 hard routing (unit-tested below).

## ⚠️ Run in this order
1. **`MODE = "smoke"`** (the default) → *Run All*. It takes about 10–15 minutes and ends with `SMOKE TEST PASSED`. Check the W&B group `task3-smoke`.
2. **`MODE = "full"`** → **Save Version → Save & Run All**. It takes about 2–2.5 hours; the runs appear live in the W&B group `task3`.

**Kaggle settings (checked by the pre-flight gate):** GPU **T4** · Internet **On** · Secret `WANDB_API_KEY` ticked for this notebook ·
Inputs: the Pet dataset + **notebook 00 output** + **Task 2 output (required)** + Task 1 v2 output (recommended, for the comparison).

| Section | Content |
|---|---|
| 0–1 | Setup, W&B, data, loading the Task 2 components, **pre-flight gate** |
| 2 | Unit tests (mixture maths, freezing, τ→0 equals hard routing, ONNX) |
| 3 | Initialised system before joint training (reference point) |
| 4 | Optuna: joint lr, τ, λ_c, λ_b, reconstruction weighting (λ₁ = α, λ_s = 1−α); pruning of poor trials and **routing collapse** |
| 5 | Final training: warm-up → joint fine-tuning (W&B per epoch, including the full 4×4 routing matrix) |
| 6 | Test set: soft MoE vs hard routing (predicted / oracle) vs universal AE, by condition × severity |
| 7 | Gate behaviour: routing heatmap, weight distributions, inactive / dominant expert checks, dominant vs distributed examples, failures |
| 8 | Stress test: **compound corruptions** (two corruptions at once), the case hard routing cannot handle |
| 9 | ONNX export of the complete soft-MoE pipeline (parity, latency, sidecar JSON) |
| 10 | Summary |

## 0. Setup

In [ ]:
import importlib.util, subprocess, sys
missing = [p for p in ("onnx", "onnxruntime", "optuna", "sklearn") if importlib.util.find_spec(p) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *[("scikit-learn" if m == "sklearn" else m) for m in missing]], check=False)
print("installed:", missing or "nothing needed")

In [ ]:
%%writefile pet_data.py
"""
pet_data.py — shared data pipeline for Tasks 1–3 (Oxford-IIIT Pet restoration).

Single source of truth for:
  * locating the Oxford-IIIT Pet dataset (Kaggle / torchvision layouts),
  * loading + standardising clean images (RGB, 128x128) with an on-disk cache
    of the *clean* resized images only (no corrupted copies are ever written),
  * the deterministic 80/20 train/val split of the official `trainval` list (seed 42),
  * the four input conditions and their corruption functions (pure NumPy, so the
    FastAPI backend can import them without PyTorch),
  * validation / test corruption manifests (deterministic, fully parameterised),
  * PyTorch datasets, a class-balanced batch sampler and DataLoader builders.

Label convention (used everywhere, incl. the Task 3 gate weight order w0..w3):
    0 = clean, 1 = salt_pepper, 2 = gaussian_blur, 3 = occlusion
"""
from __future__ import annotations

import hashlib
import json
import os
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from typing import Iterable, Sequence

import numpy as np
from PIL import Image

# --------------------------------------------------------------------------------------
# Constants (the assignment's corruption specification)
# --------------------------------------------------------------------------------------
IMG_SIZE = 128
SPLIT_SEED = 42
VAL_FRACTION = 0.20

CLEAN, SALT_PEPPER, BLUR, OCCLUSION = 0, 1, 2, 3
CLASS_NAMES = ("clean", "salt_pepper", "gaussian_blur", "occlusion")
NUM_CLASSES = 4

SP_PROB_RANGE = (0.02, 0.15)          # uniform
BLUR_KERNELS = (3, 5, 7)              # uniform choice
BLUR_SIGMA_RANGE = (0.5, 2.5)         # uniform
OCC_BOX_RANGE = (1, 3)                # inclusive, uniform
OCC_COVERAGE_RANGE = (0.10, 0.35)     # union of boxes / image area, uniform
OCC_ASPECT_RANGE = (0.5, 2.0)         # box width/height, log-uniform (design choice)
OCC_MIN_SIDE = 4                      # px (design choice)
TEST_OCC_TOLERANCE = 0.01             # |actual - target| coverage for fixed test boxes

SEVERITY_LEVELS = ("low", "medium", "high")
TEST_SEVERITIES = {
    SALT_PEPPER: {"low": {"p": 0.03}, "medium": {"p": 0.08}, "high": {"p": 0.15}},
    BLUR: {"low": {"kernel_size": 3, "sigma": 0.7},
           "medium": {"kernel_size": 5, "sigma": 1.5},
           "high": {"kernel_size": 7, "sigma": 2.5}},
    OCCLUSION: {"low": {"n_boxes": 1, "coverage": 0.10},
                "medium": {"n_boxes": 2, "coverage": 0.20},
                "high": {"n_boxes": 3, "coverage": 0.35}},
}

MANIFEST_VERSION = 1


def corruption_config() -> dict:
    """The full corruption configuration as a JSON-serialisable dict (for W&B / report)."""
    return {
        "image_size": IMG_SIZE,
        "class_names": list(CLASS_NAMES),
        "train_class_probs": [0.25] * 4,
        "salt_pepper": {"p_uniform": list(SP_PROB_RANGE), "salt_vs_pepper": 0.5,
                        "granularity": "per-pixel (all 3 channels)"},
        "gaussian_blur": {"kernel_sizes": list(BLUR_KERNELS), "sigma_uniform": list(BLUR_SIGMA_RANGE),
                          "padding": "reflect", "separable": True},
        "occlusion": {"n_boxes": list(OCC_BOX_RANGE), "coverage_uniform": list(OCC_COVERAGE_RANGE),
                      "aspect_log_uniform": list(OCC_ASPECT_RANGE), "min_side_px": OCC_MIN_SIDE,
                      "boxes_overlap": False, "fill": 0.0},
        "test_severities": {CLASS_NAMES[k]: v for k, v in TEST_SEVERITIES.items()},
        "test_occlusion_tolerance": TEST_OCC_TOLERANCE,
    }


# --------------------------------------------------------------------------------------
# Dataset discovery and loading
# --------------------------------------------------------------------------------------
def read_split_list(path: str | Path) -> list[str]:
    """Read an official list file (e.g. 'Abyssinian_100 1 1 1'). Breed/species ids are discarded."""
    names = []
    for line in Path(path).read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#"):
            names.append(line.split()[0])
    return names


def find_pet_dataset(search_root: str | Path = "/kaggle/input") -> dict:
    """Locate annotations (trainval.txt + test.txt) and the images folder under `search_root`.

    Works for the common Kaggle upload (images/images, annotations/annotations) and the
    torchvision layout (oxford-iiit-pet/images, oxford-iiit-pet/annotations).
    """
    search_root = Path(search_root)
    ann_dir = None
    for tv in sorted(search_root.rglob("trainval.txt")):
        if (tv.parent / "test.txt").exists():
            ann_dir = tv.parent
            break
    if ann_dir is None:
        raise FileNotFoundError(
            f"No folder containing both trainval.txt and test.txt under {search_root}. "
            "The official split lists are required; set IMAGES_DIR / ANNOT_DIR manually.")
    first = read_split_list(ann_dir / "trainval.txt")[0]
    hits = sorted(search_root.rglob(f"{first}.jpg"))
    if not hits:
        raise FileNotFoundError(f"Could not find image '{first}.jpg' under {search_root}.")
    return {"annotations_dir": ann_dir, "images_dir": hits[0].parent}


def load_standardised_image(path: str | Path, size: int = IMG_SIZE) -> tuple[np.ndarray, str]:
    """Open any image, force 3-channel RGB, resize to size x size (bicubic, antialiased).

    Returns (uint8 HxWx3 array, original PIL mode). The backend must use this same function.
    """
    with Image.open(path) as im:
        mode = im.mode
        im = im.convert("RGB").resize((size, size), Image.Resampling.BICUBIC)
        return np.asarray(im, dtype=np.uint8).copy(), mode


def load_clean_images(names: Sequence[str], images_dir: str | Path, cache_path: str | Path | None = None,
                      size: int = IMG_SIZE, num_threads: int = 8, verbose: bool = True) -> dict:
    """Load + standardise every image in `names`. Caches the CLEAN resized uint8 array to .npz.

    Returns dict(images=(N,H,W,3) uint8, names=[...kept...], failed=[...], modes=Counter).
    Images that cannot be decoded are reported and dropped (the split is made afterwards).
    """
    names = list(names)
    if cache_path is not None and Path(cache_path).exists():
        z = np.load(cache_path, allow_pickle=False)
        if list(z["requested"]) == names and int(z["size"]) == size:
            out = {"images": z["images"], "names": list(z["names"]), "failed": list(z["failed"]),
                   "modes": Counter(json.loads(str(z["modes"]))), "from_cache": True}
            if verbose:
                print(f"Loaded {len(out['names'])} cached images from {cache_path}")
            return out

    images_dir = Path(images_dir)

    def _load(name):
        try:
            arr, mode = load_standardised_image(images_dir / f"{name}.jpg", size)
            return name, arr, mode, None
        except Exception as e:  # corrupted / unreadable file
            return name, None, None, repr(e)

    with ThreadPoolExecutor(max_workers=num_threads) as ex:
        results = list(ex.map(_load, names))

    kept = [(n, a, m) for n, a, m, err in results if err is None]
    failed = [(n, err) for n, a, m, err in results if err is not None]
    imgs = np.stack([a for _, a, _ in kept]) if kept else np.zeros((0, size, size, 3), np.uint8)
    out = {"images": imgs, "names": [n for n, _, _ in kept], "failed": [n for n, _ in failed],
           "modes": Counter(m for _, _, m in kept), "from_cache": False}
    if verbose:
        print(f"Loaded {len(kept)}/{len(names)} images; failed: {len(failed)}")
        for n, err in failed:
            print(f"  ! {n}: {err}")
    if cache_path is not None:
        Path(cache_path).parent.mkdir(parents=True, exist_ok=True)
        np.savez(cache_path, images=imgs, names=np.array(out["names"]), requested=np.array(names),
                 failed=np.array(out["failed"], dtype=str), size=size, modes=json.dumps(dict(out["modes"])))
    return out


# --------------------------------------------------------------------------------------
# Deterministic split
# --------------------------------------------------------------------------------------
def make_split(names: Sequence[str], val_fraction: float = VAL_FRACTION, seed: int = SPLIT_SEED) -> dict:
    """80/20 random split of the official trainval list with numpy default_rng(seed)."""
    names = list(names)
    n = len(names)
    perm = np.random.default_rng(seed).permutation(n)
    n_val = int(round(val_fraction * n))
    n_train = n - n_val
    train = [names[i] for i in sorted(perm[:n_train])]
    val = [names[i] for i in sorted(perm[n_train:])]
    return {"seed": seed, "val_fraction": val_fraction, "method": "numpy.default_rng(seed).permutation",
            "n_total": n, "train": train, "val": val}


# --------------------------------------------------------------------------------------
# Corruption functions (pure NumPy; img = float32 HxWx3 in [0,1])
# --------------------------------------------------------------------------------------
def salt_and_pepper(img: np.ndarray, p: float, seed: int) -> np.ndarray:
    """Each pixel is selected with prob p; selected pixels become white or black (50/50)."""
    rng = np.random.default_rng(seed)
    h, w = img.shape[:2]
    selected = rng.random((h, w)) < p
    salt = rng.random((h, w)) < 0.5
    out = img.copy()
    out[selected & salt] = 1.0
    out[selected & ~salt] = 0.0
    return out


def gaussian_kernel1d(kernel_size: int, sigma: float) -> np.ndarray:
    x = np.arange(kernel_size, dtype=np.float64) - (kernel_size - 1) / 2.0
    k = np.exp(-0.5 * (x / sigma) ** 2)
    return (k / k.sum()).astype(np.float32)


def gaussian_blur(img: np.ndarray, kernel_size: int, sigma: float) -> np.ndarray:
    """Separable Gaussian blur with reflect padding (matches torchvision's gaussian_blur)."""
    k = gaussian_kernel1d(kernel_size, sigma)
    r = kernel_size // 2
    h, w = img.shape[:2]
    padded = np.pad(img, ((r, r), (r, r), (0, 0)), mode="reflect")
    horiz = sum(k[i] * padded[:, i:i + w] for i in range(kernel_size))        # (h+2r, w, 3)
    out = sum(k[i] * horiz[i:i + h] for i in range(kernel_size))              # (h, w, 3)
    return np.clip(out, 0.0, 1.0).astype(np.float32)


def occlude(img: np.ndarray, boxes: Iterable[Sequence[int]], fill: float = 0.0) -> np.ndarray:
    """Black out boxes given as [x0, y0, x1, y1] (x1/y1 exclusive)."""
    out = img.copy()
    for x0, y0, x1, y1 in boxes:
        out[y0:y1, x0:x1] = fill
    return out


def boxes_coverage(boxes, h: int = IMG_SIZE, w: int = IMG_SIZE) -> float:
    mask = np.zeros((h, w), bool)
    for x0, y0, x1, y1 in boxes:
        mask[y0:y1, x0:x1] = True
    return float(mask.mean())


def sample_occlusion_boxes(rng: np.random.Generator, n_boxes: int, coverage: float,
                           h: int = IMG_SIZE, w: int = IMG_SIZE,
                           accept=None, max_attempts: int = 1000) -> tuple[list[list[int]], float]:
    """Sample `n_boxes` non-overlapping boxes whose union covers ~`coverage` of the image.

    The total area is split between boxes (each gets at least half of an equal share), each
    box gets a log-uniform aspect ratio, and boxes are placed at random non-overlapping
    positions. Non-overlap makes the union area equal to the sum of box areas, so the jointly
    covered fraction is controlled precisely. `accept(actual)->bool` decides if rounding is OK.
    """
    if accept is None:
        accept = lambda a: abs(a - coverage) <= TEST_OCC_TOLERANCE  # noqa: E731
    total = coverage * h * w
    lo_ar, hi_ar = np.log(OCC_ASPECT_RANGE[0]), np.log(OCC_ASPECT_RANGE[1])
    for _ in range(max_attempts):
        shares = 0.5 / n_boxes + 0.5 * rng.dirichlet([2.0] * n_boxes)
        boxes, ok = [], True
        for area in shares * total:
            ar = float(np.exp(rng.uniform(lo_ar, hi_ar)))
            bw = int(np.clip(round(np.sqrt(area * ar)), OCC_MIN_SIDE, w))
            bh = int(np.clip(round(area / bw), OCC_MIN_SIDE, h))
            for _try in range(200):
                x0 = int(rng.integers(0, w - bw + 1))
                y0 = int(rng.integers(0, h - bh + 1))
                cand = [x0, y0, x0 + bw, y0 + bh]
                if all(cand[2] <= b[0] or b[2] <= cand[0] or cand[3] <= b[1] or b[3] <= cand[1]
                       for b in boxes):
                    boxes.append(cand)
                    break
            else:
                ok = False
                break
        if ok:
            actual = boxes_coverage(boxes, h, w)
            if accept(actual):
                return boxes, actual
    raise RuntimeError(f"Could not place {n_boxes} boxes covering {coverage:.3f}")


def _new_seed(rng: np.random.Generator) -> int:
    return int(rng.integers(0, 2**31 - 1))


def sample_params(label: int, rng: np.random.Generator) -> dict:
    """Sample corruption parameters from the TRAINING distribution. Fully reproducible via 'seed'."""
    seed = _new_seed(rng)
    r = np.random.default_rng(seed)
    label = int(label)
    if label == CLEAN:
        return {"label": CLEAN, "type": "clean", "seed": seed}
    if label == SALT_PEPPER:
        return {"label": SALT_PEPPER, "type": "salt_pepper", "seed": seed,
                "p": float(r.uniform(*SP_PROB_RANGE))}
    if label == BLUR:
        return {"label": BLUR, "type": "gaussian_blur", "seed": seed,
                "kernel_size": int(r.choice(BLUR_KERNELS)), "sigma": float(r.uniform(*BLUR_SIGMA_RANGE))}
    if label == OCCLUSION:
        n = int(r.integers(OCC_BOX_RANGE[0], OCC_BOX_RANGE[1] + 1))
        cov = float(r.uniform(*OCC_COVERAGE_RANGE))
        lo, hi = OCC_COVERAGE_RANGE
        boxes, actual = sample_occlusion_boxes(r, n, cov, accept=lambda a: lo <= a <= hi)
        return {"label": OCCLUSION, "type": "occlusion", "seed": seed, "n_boxes": n,
                "coverage_target": cov, "coverage": actual, "boxes": boxes}
    raise ValueError(f"Unknown label {label}")


def fixed_params(label: int, level: str, seed: int) -> dict:
    """Parameters for the fixed TEST severities (low/medium/high)."""
    label = int(label)
    if label == CLEAN:
        return {"label": CLEAN, "type": "clean", "seed": seed}
    spec = TEST_SEVERITIES[label][level]
    base = {"label": label, "type": CLASS_NAMES[label], "seed": seed}
    if label == SALT_PEPPER:
        return {**base, "p": spec["p"]}
    if label == BLUR:
        return {**base, "kernel_size": spec["kernel_size"], "sigma": spec["sigma"]}
    boxes, actual = sample_occlusion_boxes(np.random.default_rng(seed), spec["n_boxes"], spec["coverage"])
    return {**base, "n_boxes": spec["n_boxes"], "coverage_target": spec["coverage"],
            "coverage": actual, "boxes": boxes}


def apply_corruption(img: np.ndarray, params: dict) -> np.ndarray:
    """Deterministically apply a parameter dict (from sample_params / fixed_params / a manifest)."""
    label = int(params["label"])
    if label == CLEAN:
        return img.copy()
    if label == SALT_PEPPER:
        return salt_and_pepper(img, params["p"], params["seed"])
    if label == BLUR:
        return gaussian_blur(img, params["kernel_size"], params["sigma"])
    if label == OCCLUSION:
        return occlude(img, params["boxes"])
    raise ValueError(f"Unknown label {label}")


def describe_params(params: dict) -> str:
    """Short human-readable description (plot titles, API responses)."""
    t = int(params["label"])
    if t == CLEAN:
        return "clean"
    if t == SALT_PEPPER:
        return f"s&p p={params['p']:.3f}"
    if t == BLUR:
        return f"blur k={params['kernel_size']} σ={params['sigma']:.2f}"
    return f"occl {params['n_boxes']}box {100 * params['coverage']:.1f}%"


# --------------------------------------------------------------------------------------
# Manifests
# --------------------------------------------------------------------------------------
def build_val_manifest(val_names: Sequence[str], seed: int = SPLIT_SEED) -> dict:
    """Exactly balanced (25% per condition) validation manifest; params from the training ranges."""
    rng = np.random.default_rng(np.random.SeedSequence([seed, 1]))
    n = len(val_names)
    labels = rng.permutation(np.resize(np.arange(NUM_CLASSES), n))
    records = []
    for i, (name, label) in enumerate(zip(val_names, labels)):
        p = sample_params(int(label), rng)
        records.append({"id": f"{name}__{p['type']}", "image": name, "index": i,
                        "severity_level": "sampled" if label != CLEAN else "none", **p})
    return {"meta": {"kind": "validation", "version": MANIFEST_VERSION, "seed": seed,
                     "seed_sequence": [seed, 1], "n_images": n, "n_records": len(records),
                     "config": corruption_config()},
            "records": records}


def build_test_manifest(test_names: Sequence[str], seed: int = SPLIT_SEED) -> dict:
    """Every test image × {clean, 3 corruptions × 3 fixed severities} = 10 records per image."""
    rng = np.random.default_rng(np.random.SeedSequence([seed, 2]))
    records = []
    for i, name in enumerate(test_names):
        s = _new_seed(rng)
        records.append({"id": f"{name}__clean", "image": name, "index": i,
                        "severity_level": "none", **fixed_params(CLEAN, "none", s)})
        for label in (SALT_PEPPER, BLUR, OCCLUSION):
            for level in SEVERITY_LEVELS:
                s = _new_seed(rng)
                p = fixed_params(label, level, s)
                records.append({"id": f"{name}__{p['type']}__{level}", "image": name, "index": i,
                                "severity_level": level, **p})
    return {"meta": {"kind": "test", "version": MANIFEST_VERSION, "seed": seed,
                     "seed_sequence": [seed, 2], "n_images": len(test_names), "n_records": len(records),
                     "config": corruption_config()},
            "records": records}


def save_json(obj, path: str | Path, indent=None) -> str:
    """Write JSON deterministically and return its SHA-256 (use it to prove files are identical)."""
    text = json.dumps(obj, indent=indent, sort_keys=True, separators=None if indent else (",", ":"))
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(text)
    return hashlib.sha256(text.encode()).hexdigest()


def load_json(path: str | Path):
    return json.loads(Path(path).read_text())


def sha256_file(path: str | Path) -> str:
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


# --------------------------------------------------------------------------------------
# PyTorch datasets / samplers / loaders  (torch imported lazily so the backend stays light)
# --------------------------------------------------------------------------------------
try:
    import torch
    from torch.utils.data import DataLoader, Dataset, Sampler, get_worker_info
except ImportError:  # backend without torch: only the NumPy functions above are needed
    torch = None
    Dataset = object
    Sampler = object


def to_chw_tensor(img_hwc: np.ndarray):
    return torch.from_numpy(np.ascontiguousarray(img_hwc.transpose(2, 0, 1), dtype=np.float32))


class RuntimeCorruptionDataset(Dataset):
    """Training dataset: a NEW corruption type + severity is sampled on every __getitem__.

    Returns (corrupted_image, clean_image, corruption_label):
        (3,128,128) float32 [0,1], (3,128,128) float32 [0,1], int64 scalar.
    `classes` restricts which conditions are drawn (e.g. (1,) for the Task 2 salt-and-pepper
    specialist); they are drawn uniformly. `item` may also be an (index, label) tuple, which is
    how BalancedCorruptionBatchSampler forces exactly balanced batches.
    """

    def __init__(self, images_u8: np.ndarray, classes: Sequence[int] = (0, 1, 2, 3), seed: int | None = None):
        self.images = images_u8
        self.classes = np.asarray(classes, dtype=np.int64)
        self.seed = seed
        self._rng, self._rng_key = None, None

    def __len__(self):
        return len(self.images)

    def _get_rng(self) -> np.random.Generator:
        # One independent generator per process/worker. Worker seeds come from torch
        # (base_seed + worker_id, re-drawn every epoch), so torch.manual_seed controls everything
        # and forked workers never share a random stream.
        info = get_worker_info()
        key = (os.getpid(), info.seed if info is not None else None)
        if key != self._rng_key:
            if info is not None:
                s = info.seed % 2**32
            else:
                s = self.seed if self.seed is not None else int(torch.initial_seed() % 2**32)
            self._rng, self._rng_key = np.random.default_rng(s), key
        return self._rng

    def get_with_params(self, item):
        rng = self._get_rng()
        if isinstance(item, (tuple, list)):
            idx, label = int(item[0]), int(item[1])
        else:
            idx, label = int(item), int(rng.choice(self.classes))
        clean = self.images[idx].astype(np.float32) / 255.0
        params = sample_params(label, rng)
        corrupted = apply_corruption(clean, params)
        return corrupted, clean, params

    def __getitem__(self, item):
        corrupted, clean, params = self.get_with_params(item)
        return to_chw_tensor(corrupted), to_chw_tensor(clean), torch.tensor(params["label"], dtype=torch.long)


class ManifestCorruptionDataset(Dataset):
    """Validation/test dataset: corruption parameters are read from a fixed manifest.

    `name_to_row` maps image name -> row in `images_u8`. With return_index=True a 4th element
    (the record index) is returned so results can be grouped by type/severity afterwards.
    """

    def __init__(self, images_u8: np.ndarray, name_to_row: dict, records: list[dict],
                 classes: Sequence[int] | None = None, return_index: bool = False):
        self.images = images_u8
        self.name_to_row = name_to_row
        self.records = [r for r in records if classes is None or r["label"] in classes]
        self.return_index = return_index

    def __len__(self):
        return len(self.records)

    def __getitem__(self, i):
        rec = self.records[i]
        clean = self.images[self.name_to_row[rec["image"]]].astype(np.float32) / 255.0
        corrupted = apply_corruption(clean, rec)
        out = (to_chw_tensor(corrupted), to_chw_tensor(clean), torch.tensor(rec["label"], dtype=torch.long))
        return out + (torch.tensor(i),) if self.return_index else out


class BalancedCorruptionBatchSampler(Sampler):
    """Yields batches of (image_index, label) with EXACTLY equal counts per class
    (when batch_size is a multiple of the number of classes). Intended for the Task 2 classifier.
    """

    def __init__(self, n_items: int, batch_size: int, classes: Sequence[int] = (0, 1, 2, 3),
                 drop_last: bool = True, seed: int | None = None):
        self.n, self.bs, self.classes, self.drop_last = n_items, batch_size, np.asarray(classes), drop_last
        self.seed, self.epoch = seed, 0

    def __len__(self):
        return self.n // self.bs if self.drop_last else -(-self.n // self.bs)

    def __iter__(self):
        base = self.seed if self.seed is not None else int(torch.randint(0, 2**31 - 1, ()).item())
        rng = np.random.default_rng([base, self.epoch])
        self.epoch += 1
        perm = rng.permutation(self.n)
        for start in range(0, len(self) * self.bs, self.bs):
            idx = perm[start:start + self.bs]
            labels = rng.permutation(np.resize(self.classes, len(idx)))
            yield [(int(i), int(l)) for i, l in zip(idx, labels)]


def default_num_workers() -> int:
    return max(0, min(4, (os.cpu_count() or 2)))


def make_train_loader(images_u8, batch_size=32, classes=(0, 1, 2, 3), balanced=False,
                      num_workers=None, seed=None, pin_memory=True, drop_last=True):
    num_workers = default_num_workers() if num_workers is None else num_workers
    ds = RuntimeCorruptionDataset(images_u8, classes=classes, seed=seed)
    kw = dict(num_workers=num_workers, pin_memory=pin_memory and torch.cuda.is_available(),
              persistent_workers=num_workers > 0)
    if balanced:
        bs = BalancedCorruptionBatchSampler(len(ds), batch_size, classes, drop_last=drop_last, seed=seed)
        return DataLoader(ds, batch_sampler=bs, **kw)
    return DataLoader(ds, batch_size=batch_size, shuffle=True, drop_last=drop_last, **kw)


def make_eval_loader(images_u8, name_to_row, records, batch_size=64, classes=None,
                     return_index=False, num_workers=None, pin_memory=True):
    num_workers = default_num_workers() if num_workers is None else num_workers
    ds = ManifestCorruptionDataset(images_u8, name_to_row, records, classes=classes, return_index=return_index)
    # Not persistent: eval workers exit after each pass, so idle loaders don't hold RAM
    # (each worker gradually copies the Python objects it touches — "copy-on-read").
    return DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=num_workers,
                      pin_memory=pin_memory and torch.cuda.is_available(), persistent_workers=False)


class PetData:
    """Everything a Task 1–3 notebook needs, built by prepare_data()."""

    def __init__(self, **kw):
        self.__dict__.update(kw)

    def load_test_images(self, cache_dir=None):
        """FINAL EVALUATION ONLY: decode the quarantined official test images (cached)."""
        cache = Path(cache_dir or self.cache_dir) / "test_128.npz"
        out = load_clean_images(self.test_names, self.images_dir, cache_path=cache)
        return out["images"], {n: i for i, n in enumerate(out["names"])}, out["failed"]

    def summary(self) -> dict:
        return {"source": self.source, "images_dir": str(self.images_dir), "n_train": len(self.train_images),
                "n_val": len(self.val_images), "n_test": len(self.test_names),
                "n_val_records": len(self.val_manifest["records"]),
                "n_test_records": len(self.test_manifest["records"]),
                "split_sha256": self.hashes.get("split"), "val_manifest_sha256": self.hashes.get("val"),
                "test_manifest_sha256": self.hashes.get("test")}


def find_prep_outputs(search_root="/kaggle/input") -> Path | None:
    """Locate an attached output of notebook 00 (folder containing manifests/val_manifest.json)."""
    for p in sorted(Path(search_root).rglob("val_manifest.json")):
        root = p.parent.parent
        if (root / "manifests" / "test_manifest.json").exists() and (root / "artifacts" / "split_seed42.json").exists():
            return root
    return None


def prepare_data(input_root="/kaggle/input", work_dir="/kaggle/working", images_dir=None, annot_dir=None,
                 seed: int = SPLIT_SEED, verbose: bool = True) -> PetData:
    """Return split, clean train/val arrays and both manifests.

    1. If notebook 00's output is attached under `input_root`, its split + manifests are reused
       (guarantees byte-identical data across Tasks 1–3).
    2. Otherwise everything is regenerated deterministically (same seeds -> same files/hashes)
       and saved under `work_dir`.
    The official Oxford-IIIT Pet dataset must be attached in both cases (for the pixels).
    """
    input_root, work_dir = Path(input_root), Path(work_dir)
    cache_dir = work_dir / "cache"
    if images_dir is None or annot_dir is None:
        found = find_pet_dataset(input_root)
        images_dir = Path(images_dir or found["images_dir"])
        annot_dir = Path(annot_dir or found["annotations_dir"])
    test_names = read_split_list(Path(annot_dir) / "test.txt")

    prep = find_prep_outputs(input_root)
    if prep is not None:
        source = f"attached notebook-00 output: {prep}"
        split = load_json(prep / "artifacts" / "split_seed42.json")
        val_manifest = load_json(prep / "manifests" / "val_manifest.json")
        test_manifest = load_json(prep / "manifests" / "test_manifest.json")
        hashes = {"split": sha256_file(prep / "artifacts" / "split_seed42.json"),
                  "val": sha256_file(prep / "manifests" / "val_manifest.json"),
                  "test": sha256_file(prep / "manifests" / "test_manifest.json")}
        wanted = split["train"] + split["val"]
        cached = prep / "cache" / "trainval_128.npz"
        tv = None
        if cached.exists():
            z = load_clean_images(read_split_list(Path(annot_dir) / "trainval.txt"), images_dir,
                                  cache_path=cached, verbose=False)
            tv = z if set(wanted) <= set(z["names"]) else None
        if tv is None:
            tv = load_clean_images(wanted, images_dir, cache_path=cache_dir / "trainval_128.npz", verbose=verbose)
    else:
        source = "regenerated deterministically (notebook-00 output not attached)"
        trainval_names = read_split_list(Path(annot_dir) / "trainval.txt")
        tv = load_clean_images(trainval_names, images_dir, cache_path=cache_dir / "trainval_128.npz", verbose=verbose)
        split = make_split(tv["names"], VAL_FRACTION, seed)
        split["excluded_unreadable"] = tv["failed"]
        val_manifest = build_val_manifest(split["val"], seed)
        test_manifest = build_test_manifest(test_names, seed)
        hashes = {"split": save_json(split, work_dir / "artifacts" / "split_seed42.json", indent=1),
                  "val": save_json(val_manifest, work_dir / "manifests" / "val_manifest.json"),
                  "test": save_json(test_manifest, work_dir / "manifests" / "test_manifest.json")}

    row = {n: i for i, n in enumerate(tv["names"])}
    train_images = tv["images"][[row[n] for n in split["train"]]]
    val_images = tv["images"][[row[n] for n in split["val"]]]
    assert not set(split["train"]) & set(split["val"]) and not (set(split["train"]) | set(split["val"])) & set(test_names)
    data = PetData(source=source, images_dir=images_dir, annot_dir=annot_dir, cache_dir=cache_dir, split=split,
                   train_images=train_images, val_images=val_images,
                   val_name_to_row={n: i for i, n in enumerate(split["val"])},
                   val_manifest=val_manifest, test_manifest=test_manifest, test_names=test_names, hashes=hashes)
    if verbose:
        for k, v in data.summary().items():
            print(f"  {k:22s} {v}")
    return data


def seed_everything(seed: int = SPLIT_SEED):
    import random
    random.seed(seed)
    np.random.seed(seed)
    if torch is not None:
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

In [ ]:
%%writefile restoration.py
"""
restoration.py — shared restoration components (Task 1, reused by Task 2 specialists and Task 3).

Contents
  * SSIM (Gaussian window, Wang et al. 2004) + PSNR, per-image and batched, differentiable
  * L1 + SSIM loss:  L = alpha * L1 + (1 - alpha) * (1 - SSIM)
  * ConvAutoencoder: conv encoder -> compressed spatial latent (1x1-conv bottleneck + dropout)
                     -> conv decoder, with optional LIMITED skip connections (deepest levels only)
  * train_one_epoch / evaluate (AMP-aware, per-corruption metrics)
  * ONNX export, PyTorch-vs-ONNX parity check, latency measurement
  * plotting helpers for restoration grids (target / input / output / |error|)
"""
from __future__ import annotations

import math
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

CLASS_NAMES = ("clean", "salt_pepper", "gaussian_blur", "occlusion")


# ======================================================================================
# Metrics
# ======================================================================================
def _gaussian_window(size: int, sigma: float, channels: int, device, dtype):
    x = torch.arange(size, device=device, dtype=dtype) - (size - 1) / 2
    g = torch.exp(-(x ** 2) / (2 * sigma ** 2))
    g = g / g.sum()
    w2d = g[:, None] * g[None, :]
    return w2d.expand(channels, 1, size, size).contiguous()


def ssim_map(x: torch.Tensor, y: torch.Tensor, window_size: int = 11, sigma: float = 1.5,
             data_range: float = 1.0) -> torch.Tensor:
    """SSIM map (B,C,H',W') with an 11x11 Gaussian window (sigma 1.5), 'valid' region only.
    Matches skimage.structural_similarity(gaussian_weights=True, sigma=1.5,
    use_sample_covariance=False, data_range=1)."""
    x, y = x.float(), y.float()
    c = x.shape[1]
    w = _gaussian_window(window_size, sigma, c, x.device, x.dtype)
    c1, c2 = (0.01 * data_range) ** 2, (0.03 * data_range) ** 2
    mu_x = F.conv2d(x, w, groups=c)
    mu_y = F.conv2d(y, w, groups=c)
    sxx = F.conv2d(x * x, w, groups=c) - mu_x ** 2
    syy = F.conv2d(y * y, w, groups=c) - mu_y ** 2
    sxy = F.conv2d(x * y, w, groups=c) - mu_x * mu_y
    return ((2 * mu_x * mu_y + c1) * (2 * sxy + c2)) / ((mu_x ** 2 + mu_y ** 2 + c1) * (sxx + syy + c2))


def ssim_per_image(x, y, **kw) -> torch.Tensor:
    return ssim_map(x, y, **kw).mean(dim=(1, 2, 3))


def psnr_per_image(x, y, data_range: float = 1.0) -> torch.Tensor:
    mse = ((x.float() - y.float()) ** 2).mean(dim=(1, 2, 3)).clamp_min(1e-10)
    return 10 * torch.log10(data_range ** 2 / mse)


def mae_per_image(x, y) -> torch.Tensor:
    return (x.float() - y.float()).abs().mean(dim=(1, 2, 3))


def val_score(psnr_mean: float, ssim_mean: float) -> float:
    """Optuna objective (alpha-independent, so the study cannot 'win' by changing the loss scale):
    equal-weight combination of structural similarity and reconstruction fidelity (PSNR/40, capped)."""
    return 0.5 * float(ssim_mean) + 0.5 * min(float(psnr_mean) / 40.0, 1.0)


# ======================================================================================
# Loss
# ======================================================================================
class L1SSIMLoss(nn.Module):
    """alpha * L1 + (1 - alpha) * (1 - SSIM). Always computed in float32 (AMP-safe)."""

    def __init__(self, alpha: float = 0.8):
        super().__init__()
        self.alpha = float(alpha)

    def forward(self, pred, target):
        pred, target = pred.float(), target.float()
        l1 = F.l1_loss(pred, target)
        s = ssim_map(pred, target).mean()
        total = self.alpha * l1 + (1 - self.alpha) * (1 - s)
        return total, {"l1": l1.detach(), "ssim": s.detach()}


# ======================================================================================
# Model
# ======================================================================================
def norm(c: int) -> nn.GroupNorm:
    # GroupNorm instead of BatchNorm: no running statistics, so train- and eval-mode behave the same
    # at any batch size (Optuna samples batch sizes 16-64), and frozen experts in Task 3 cannot drift.
    return nn.GroupNorm(min(8, c), c)


def conv_block(cin: int, cout: int, stride: int = 1) -> nn.Sequential:
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, stride, 1, bias=False), norm(cout), nn.SiLU(inplace=True),
        nn.Conv2d(cout, cout, 3, 1, 1, bias=False), norm(cout), nn.SiLU(inplace=True),
    )


class ConvAutoencoder(nn.Module):
    """Convolutional denoising autoencoder with a genuine compressed code.

    Encoder: 128 -> 64 -> 32 -> 16 -> 8 (stride-2 conv blocks), channels grow base x (1,2,4,8,8).
    Latent:  1x1 conv to `latent_channels` at 8x8 (e.g. 16 ch -> 1,024 values), then code dropout.
    Decoder: bilinear upsample + conv blocks (no transposed-conv checkerboard artefacts), sigmoid output.

    Skip connections (skip_levels = 0, 1 or 2) are BOTTLENECKED: the encoder feature at the
    16x16 (and 32x32) level is squeezed by a 1x1 conv to `skip_channels` channels and passes the same
    code dropout before reaching the decoder. The information the decoder receives is therefore
    bounded by the total code size

        code_dim = latent_dim + sum_over_skipped_levels(skip_channels * H_l * W_l),

    which is what `compression_ratio` reports (input values / code values). Full- and half-resolution
    features are never skipped. skip_channels=None reproduces the original full-width skips
    (e.g. 384 x 16 x 16 = 98,304 values, larger than the image); it is kept only for the ablation,
    which showed that such skips let the decoder ignore the latent.
    """

    def __init__(self, base_channels: int = 32, latent_channels: int = 32, dropout: float = 0.1,
                 skip_levels: int = 0, skip_channels: int | None = 4, depth: int = 4, max_mult: int = 8,
                 img_size: int = 128):
        super().__init__()
        assert 0 <= skip_levels <= 2, "only the two deepest levels may be skipped"
        self.config = dict(base_channels=base_channels, latent_channels=latent_channels, dropout=dropout,
                           skip_levels=skip_levels, skip_channels=skip_channels, depth=depth,
                           max_mult=max_mult, img_size=img_size)
        chs = [min(base_channels * 2 ** i, base_channels * max_mult) for i in range(depth + 1)]
        self.chs, self.depth, self.skip_levels, self.img_size = chs, depth, skip_levels, img_size
        self.stem = conv_block(3, chs[0], 1)
        self.down = nn.ModuleList([conv_block(chs[i], chs[i + 1], 2) for i in range(depth)])
        self.to_latent = nn.Conv2d(chs[-1], latent_channels, 1)
        self.code_dropout = nn.Dropout(dropout)
        self.from_latent = nn.Sequential(nn.Conv2d(latent_channels, chs[-1], 1, bias=False),
                                         norm(chs[-1]), nn.SiLU(inplace=True))
        # decoder stage j upsamples to resolution level (depth - j - 1); skips only for j < skip_levels
        self.skip_proj = nn.ModuleList()
        self.skip_shapes = []
        self.up = nn.ModuleList()
        for j in range(depth):
            lvl = depth - j - 1
            extra = 0
            if j < skip_levels:
                width = chs[lvl] if skip_channels is None else skip_channels
                self.skip_proj.append(nn.Identity() if skip_channels is None else nn.Conv2d(chs[lvl], skip_channels, 1))
                res = img_size // 2 ** lvl
                self.skip_shapes.append((width, res, res))
                extra = width
            self.up.append(conv_block(chs[lvl + 1] + extra, chs[lvl], 1))
        self.head = nn.Conv2d(chs[0], 3, 3, 1, 1)
        self.latent_shape = (latent_channels, img_size // 2 ** depth, img_size // 2 ** depth)

    # ---- code-size accounting ------------------------------------------------------------
    @property
    def latent_dim(self) -> int:
        return int(np.prod(self.latent_shape))

    @property
    def skip_dim(self) -> int:
        return int(sum(np.prod(s) for s in self.skip_shapes))

    @property
    def code_dim(self) -> int:
        """Total number of values the decoder receives from the encoder (latent + skip codes)."""
        return self.latent_dim + self.skip_dim

    @property
    def compression_ratio(self) -> float:
        return 3 * self.img_size ** 2 / self.code_dim

    # ---- forward -------------------------------------------------------------------------
    def encode(self, x):
        """Returns (latent, skip_codes). skip_codes[j] feeds decoder stage j (deepest first)."""
        feats = []
        h = self.stem(x)
        for blk in self.down:
            feats.append(h)
            h = blk(h)
        skips = [proj(feats[self.depth - j - 1]) for j, proj in enumerate(self.skip_proj)]
        return self.to_latent(h), skips

    def decode(self, z, skips=None):
        h = self.from_latent(self.code_dropout(z))
        for j, blk in enumerate(self.up):
            h = F.interpolate(h, scale_factor=2, mode="bilinear", align_corners=False)
            if j < self.skip_levels:
                s = skips[j] if skips is not None else h.new_zeros(h.shape[0], *self.skip_shapes[j])
                h = torch.cat([h, self.code_dropout(s)], dim=1)
            h = blk(h)
        return torch.sigmoid(self.head(h))

    def forward(self, x):
        z, skips = self.encode(x)
        return self.decode(z, skips)


@torch.no_grad()
def code_dependence(model, loader, device, max_batches=None) -> dict:
    """How much the output depends on each part of the code (evaluated on a manifest loader).

    full:        normal forward pass
    zero_latent: latent replaced by zeros (skips intact)   -> large drop = the latent is used
    zero_skips:  skip codes replaced by zeros (latent intact) -> drop = what the skips contribute
    """
    model.eval()
    acc = {k: {"psnr": [], "ssim": []} for k in ("full", "zero_latent", "zero_skips")}
    for b, batch in enumerate(loader):
        if max_batches is not None and b >= max_batches:
            break
        x, y = batch[0].to(device), batch[1].to(device)
        z, skips = model.encode(x)
        outs = {"full": model.decode(z, skips), "zero_latent": model.decode(torch.zeros_like(z), skips)}
        if model.skip_levels:
            outs["zero_skips"] = model.decode(z, [torch.zeros_like(s) for s in skips])
        for k, o in outs.items():
            o = o.float().clamp(0, 1)
            acc[k]["psnr"].append(psnr_per_image(o, y).cpu())
            acc[k]["ssim"].append(ssim_per_image(o, y).cpu())
    out = {}
    for k, d in acc.items():
        if d["psnr"]:
            out[f"{k}_psnr"] = torch.cat(d["psnr"]).mean().item()
            out[f"{k}_ssim"] = torch.cat(d["ssim"]).mean().item()
    out["latent_drop_db"] = out["full_psnr"] - out["zero_latent_psnr"]
    if "zero_skips_psnr" in out:
        out["skip_drop_db"] = out["full_psnr"] - out["zero_skips_psnr"]
    return out


def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# ======================================================================================
# Training / evaluation
# ======================================================================================
def make_optimizer_and_scheduler(model, lr, weight_decay, steps_per_epoch, epochs, warmup_epochs=1):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    total = max(1, steps_per_epoch * epochs)
    warm = max(1, int(steps_per_epoch * warmup_epochs))

    def lr_lambda(step):
        if step < warm:
            return (step + 1) / warm
        t = (step - warm) / max(1, total - warm)
        return 0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * min(1.0, t)))

    return opt, torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)


def make_scaler(device, amp=True):
    """One GradScaler per training run (fp16 AMP on CUDA only; a no-op on CPU)."""
    return torch.amp.GradScaler("cuda", enabled=bool(amp and device.type == "cuda"))


def train_one_epoch(model, loader, loss_fn, optimizer, scheduler, device, scaler=None, amp=True,
                    grad_clip=1.0, max_batches=None) -> dict:
    model.train()
    use_amp = amp and device.type == "cuda"
    if scaler is None:
        scaler = make_scaler(device, amp)
    tot = {"loss": 0.0, "l1": 0.0, "ssim": 0.0}
    n = 0
    for b, batch in enumerate(loader):
        if max_batches is not None and b >= max_batches:
            break
        x, y = batch[0].to(device, non_blocking=True), batch[1].to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            pred = model(x)
        loss, parts = loss_fn(pred, y)
        if not torch.isfinite(loss):
            raise FloatingPointError(f"non-finite loss at batch {b}")
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        if grad_clip:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        scaler.step(optimizer)
        scaler.update()
        if scheduler is not None:
            scheduler.step()
        bs = x.shape[0]
        tot["loss"] += loss.item() * bs
        tot["l1"] += parts["l1"].item() * bs
        tot["ssim"] += parts["ssim"].item() * bs
        n += bs
    return {k: v / max(n, 1) for k, v in tot.items()}


@torch.no_grad()
def evaluate(model, loader, device, loss_fn=None, amp=True, per_image=False, include_input=False) -> dict:
    """Metrics over a (manifest) loader. Returns means overall and per corruption type.
    per_image=True also returns numpy arrays aligned with the dataset order.
    include_input=True also scores the corrupted input itself (the 'do nothing' baseline)."""
    model.eval()
    use_amp = amp and device.type == "cuda"
    keys = ["psnr", "ssim", "mae"] + (["in_psnr", "in_ssim", "in_mae"] if include_input else [])
    acc = {k: [] for k in keys}
    labels, losses = [], []
    for batch in loader:
        x, y, lab = batch[0].to(device), batch[1].to(device), batch[2]
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            pred = model(x)
        pred = pred.float().clamp(0, 1)
        if loss_fn is not None:
            losses.append(loss_fn(pred, y)[0].item() * x.shape[0])
        acc["psnr"].append(psnr_per_image(pred, y).cpu())
        acc["ssim"].append(ssim_per_image(pred, y).cpu())
        acc["mae"].append(mae_per_image(pred, y).cpu())
        if include_input:
            acc["in_psnr"].append(psnr_per_image(x, y).cpu())
            acc["in_ssim"].append(ssim_per_image(x, y).cpu())
            acc["in_mae"].append(mae_per_image(x, y).cpu())
        labels.append(lab.cpu())
    arr = {k: torch.cat(v).numpy() for k, v in acc.items()}
    labels = torch.cat(labels).numpy()
    out = {k: float(v.mean()) for k, v in arr.items()}
    for c, name in enumerate(CLASS_NAMES):
        m = labels == c
        if m.any():
            for k, v in arr.items():
                out[f"{k}/{name}"] = float(v[m].mean())
    out["score"] = val_score(out["psnr"], out["ssim"])
    if loss_fn is not None:
        out["loss"] = sum(losses) / len(labels)
    if per_image:
        out["per_image"] = {**arr, "label": labels}
    return out


@torch.no_grad()
def predict_numpy(model, imgs_hwc: np.ndarray, device, batch_size=64) -> np.ndarray:
    """(N,H,W,3) float [0,1] -> restored (N,H,W,3) float [0,1]."""
    model.eval()
    outs = []
    for i in range(0, len(imgs_hwc), batch_size):
        x = torch.from_numpy(np.ascontiguousarray(imgs_hwc[i:i + batch_size].transpose(0, 3, 1, 2))).float().to(device)
        outs.append(model(x).float().clamp(0, 1).cpu().numpy().transpose(0, 2, 3, 1))
    return np.concatenate(outs)


# ======================================================================================
# ONNX
# ======================================================================================
def export_onnx(model, path, img_size=128, opset=17, input_name="input", output_name="output"):
    model = model.eval().cpu()
    dummy = torch.rand(1, 3, img_size, img_size)
    kw = dict(input_names=[input_name], output_names=[output_name], opset_version=opset,
              dynamic_axes={input_name: {0: "batch"}, output_name: {0: "batch"}})
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with torch.no_grad():
        try:
            torch.onnx.export(model, dummy, str(path), dynamo=False, **kw)
        except TypeError:          # torch < 2.5 has no `dynamo` kwarg
            torch.onnx.export(model, dummy, str(path), **kw)
    try:
        import onnx
        onnx.checker.check_model(onnx.load(str(path)))
    except ImportError:
        pass
    return path


def verify_onnx(model, path, x: torch.Tensor, atol=1e-4) -> dict:
    """Compare PyTorch (CPU, eval) and onnxruntime outputs on the same batch."""
    import onnxruntime as ort
    model = model.eval().cpu()
    sess = ort.InferenceSession(str(path), providers=["CPUExecutionProvider"])
    with torch.no_grad():
        ref = model(x.cpu().float()).numpy()
    out = sess.run(None, {sess.get_inputs()[0].name: x.cpu().float().numpy()})[0]
    diff = np.abs(out - ref)
    p_ref = psnr_per_image(torch.from_numpy(ref), torch.from_numpy(out))
    return {"max_abs_diff": float(diff.max()), "mean_abs_diff": float(diff.mean()),
            "min_psnr_torch_vs_onnx": float(p_ref.min()), "n_images": int(len(x)),
            "passed": bool(diff.max() < atol)}


def onnx_latency_ms(path, img_size=128, batch=1, n_warmup=10, n_runs=100) -> dict:
    import onnxruntime as ort
    sess = ort.InferenceSession(str(path), providers=["CPUExecutionProvider"])
    name = sess.get_inputs()[0].name
    x = np.random.rand(batch, 3, img_size, img_size).astype(np.float32)
    for _ in range(n_warmup):
        sess.run(None, {name: x})
    ts = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        sess.run(None, {name: x})
        ts.append((time.perf_counter() - t0) * 1000)
    return {"median_ms": float(np.median(ts)), "p90_ms": float(np.percentile(ts, 90)), "batch": batch}


# ======================================================================================
# Plotting
# ======================================================================================
def restoration_grid(rows: list[dict], path=None, title=None, err_vmax=0.5):
    """rows: dicts with 'clean', 'corrupted', 'restored' (HxWx3 float [0,1]) and optional 'label'.
    Columns: clean target | corrupted input | reconstruction | absolute error map."""
    import matplotlib.pyplot as plt
    n = len(rows)
    fig, axes = plt.subplots(n, 4, figsize=(9.2, 2.35 * n), squeeze=False)
    for i, r in enumerate(rows):
        err = np.abs(r["restored"] - r["clean"]).mean(-1)
        panels = [r["clean"], r["corrupted"], r["restored"]]
        for j, im in enumerate(panels):
            axes[i, j].imshow(np.clip(im, 0, 1))
        hm = axes[i, 3].imshow(err, cmap="magma", vmin=0, vmax=err_vmax)
        for ax in axes[i]:
            ax.set_xticks([]); ax.set_yticks([])
        axes[i, 0].set_ylabel(r.get("label", ""), fontsize=7.5, rotation=0, ha="right", va="center", labelpad=4)
        if "in_text" in r:
            axes[i, 1].set_xlabel(r["in_text"], fontsize=7.5)
        if "out_text" in r:
            axes[i, 2].set_xlabel(r["out_text"], fontsize=7.5)
    for j, t in enumerate(["clean target", "corrupted input", "reconstruction", "|error| (mean RGB)"]):
        axes[0, j].set_title(t, fontsize=9)
    fig.colorbar(hm, ax=axes[:, 3].tolist(), fraction=0.046, pad=0.02, shrink=0.6)
    if title:
        fig.suptitle(title, fontsize=11)
    if path:
        fig.savefig(path, dpi=150, bbox_inches="tight")
    return fig

In [ ]:
%%writefile classifier.py
"""
classifier.py — corruption classifier for Task 2 (reused as the gating network in Task 3).

Classes (same order everywhere, incl. the Task 3 gate weights w0..w3):
    0 = clean, 1 = salt_pepper, 2 = gaussian_blur, 3 = occlusion

Design notes
  * The first conv block runs at FULL resolution (128x128): salt-and-pepper noise and slight blur are
    pixel-level cues that are destroyed by early downsampling.
  * GroupNorm (no running statistics): train/eval behave identically, which matters when the network is
    later fine-tuned as the Task 3 gate with small batches.
  * Global AVERAGE + global MAX pooling are concatenated: max pooling responds to rare extreme activations
    (isolated black/white impulses, hard edges of black occlusion boxes); average pooling to global
    statistics (overall loss of high frequencies under blur).
  * forward() returns LOGITS; softmax / temperature are applied by the caller (Task 3: softmax(G(x)/tau)).
"""
from __future__ import annotations

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

CLASS_NAMES = ("clean", "salt_pepper", "gaussian_blur", "occlusion")
CHANNEL_CONFIGS = {
    "16-32-64-128": (16, 32, 64, 128),
    "32-64-128-256": (32, 64, 128, 256),
    "32-64-128-256-256": (32, 64, 128, 256, 256),
}


def _gn(c):
    return nn.GroupNorm(min(8, c), c)


class CorruptionClassifier(nn.Module):
    def __init__(self, channels=(32, 64, 128, 256), dropout: float = 0.2, num_classes: int = 4):
        super().__init__()
        if isinstance(channels, str):
            channels = CHANNEL_CONFIGS[channels]
        channels = tuple(int(c) for c in channels)
        self.config = dict(channels=list(channels), dropout=dropout, num_classes=num_classes)
        layers, cin = [], 3
        for i, c in enumerate(channels):
            stride = 1 if i == 0 else 2            # stage 0 keeps full resolution
            layers += [nn.Conv2d(cin, c, 3, stride, 1, bias=False), _gn(c), nn.SiLU(inplace=True),
                       nn.Conv2d(c, c, 3, 1, 1, bias=False), _gn(c), nn.SiLU(inplace=True)]
            cin = c
        self.features = nn.Sequential(*layers)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(2 * cin, num_classes)

    def forward(self, x):
        h = self.features(x)
        h = torch.cat([h.mean(dim=(2, 3)), h.amax(dim=(2, 3))], dim=1)
        return self.fc(self.dropout(h))


class ClassifierWithProbs(nn.Module):
    """ONNX wrapper: returns (logits, probabilities)."""

    def __init__(self, clf):
        super().__init__()
        self.clf = clf

    def forward(self, x):
        logits = self.clf(x)
        return logits, torch.softmax(logits, dim=1)


# --------------------------------------------------------------------------------------
# Training / evaluation
# --------------------------------------------------------------------------------------
def train_one_epoch(model, loader, optimizer, scheduler, device, scaler, amp=True, grad_clip=1.0, label_smoothing=0.0):
    model.train()
    use_amp = amp and device.type == "cuda"
    tot_loss, correct, n = 0.0, 0, 0
    for x, _, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            logits = model(x)
        loss = F.cross_entropy(logits.float(), y, label_smoothing=label_smoothing)
        if not torch.isfinite(loss):
            raise FloatingPointError("non-finite classifier loss")
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        if grad_clip:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        scaler.step(optimizer)
        scaler.update()
        if scheduler is not None:
            scheduler.step()
        tot_loss += loss.item() * len(y)
        correct += (logits.argmax(1) == y).sum().item()
        n += len(y)
    return {"loss": tot_loss / max(n, 1), "acc": correct / max(n, 1)}


@torch.no_grad()
def predict(model, loader, device, amp=True):
    """Returns (labels, probs[N,4], logits[N,4]) over a manifest loader (dataset order)."""
    model.eval()
    use_amp = amp and device.type == "cuda"
    labels, logits = [], []
    for batch in loader:
        x = batch[0].to(device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            lg = model(x)
        logits.append(lg.float().cpu())
        labels.append(batch[2])
    logits = torch.cat(logits)
    return torch.cat(labels).numpy(), torch.softmax(logits, 1).numpy(), logits.numpy()


def classification_metrics(y_true, probs) -> dict:
    """Accuracy, macro P/R/F1, per-class P/R/F1/support, row-normalised confusion matrix, CE loss."""
    from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support
    y_pred = probs.argmax(1)
    labels = list(range(len(CLASS_NAMES)))
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)
    mp, mr, mf, _ = precision_recall_fscore_support(y_true, y_pred, labels=labels, average="macro", zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    cm_norm = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    ce = float(-np.log(np.clip(probs[np.arange(len(y_true)), y_true], 1e-12, 1)).mean())
    out = {"acc": float(accuracy_score(y_true, y_pred)), "macro_precision": float(mp), "macro_recall": float(mr),
           "macro_f1": float(mf), "loss": ce, "cm": cm, "cm_norm": cm_norm}
    for i, n in enumerate(CLASS_NAMES):
        out[f"precision/{n}"], out[f"recall/{n}"], out[f"f1/{n}"], out[f"support/{n}"] = float(p[i]), float(r[i]), float(f[i]), int(s[i])
    return out


def per_class_table(m: dict):
    import pandas as pd
    rows = [{"class": n, "precision": m[f"precision/{n}"], "recall": m[f"recall/{n}"], "f1": m[f"f1/{n}"],
             "support": m[f"support/{n}"]} for n in CLASS_NAMES]
    rows.append({"class": "macro avg", "precision": m["macro_precision"], "recall": m["macro_recall"],
                 "f1": m["macro_f1"], "support": int(sum(m[f"support/{n}"] for n in CLASS_NAMES))})
    rows.append({"class": "accuracy", "precision": np.nan, "recall": np.nan, "f1": m["acc"],
                 "support": int(sum(m[f"support/{n}"] for n in CLASS_NAMES))})
    return pd.DataFrame(rows).set_index("class")


def plot_confusion(cm_norm, cm_counts=None, title="", path=None, ax=None):
    import matplotlib.pyplot as plt
    own = ax is None
    if own:
        fig, ax = plt.subplots(figsize=(4.8, 4.2))
    im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
    short = ["clean", "s&p", "blur", "occl."]
    ax.set_xticks(range(4)); ax.set_xticklabels(short); ax.set_yticks(range(4)); ax.set_yticklabels(short)
    ax.set_xlabel("predicted"); ax.set_ylabel("true")
    for i in range(4):
        for j in range(4):
            txt = f"{cm_norm[i, j]:.2f}" + (f"\n({cm_counts[i, j]})" if cm_counts is not None else "")
            ax.text(j, i, txt, ha="center", va="center", fontsize=8,
                    color="white" if cm_norm[i, j] > 0.6 else "black")
    ax.set_title(title, fontsize=10)
    if own:
        fig.colorbar(im, ax=ax, fraction=0.046)
        fig.tight_layout()
        if path:
            fig.savefig(path, dpi=150)
        return fig
    return im

In [ ]:
%%writefile moe.py
"""
moe.py — Task 3: jointly trained soft mixture-of-experts restoration.

    w      = softmax(G(x~) / tau)                          (gate G = Task 2 classifier, 4 logits)
    x_hat  = w0 * x~ + w1 * A_salt(x~) + w2 * A_blur(x~) + w3 * A_occ(x~)
    L_MoE  = l1 * L1 + ls * (1 - SSIM) + lc * CE + lb * L_balance
    L_balance = sum_k (mean_batch(w_k) - 1/4)^2           (computed on exactly balanced batches)

Branch order everywhere: 0 identity (clean), 1 salt_pepper, 2 gaussian_blur, 3 occlusion.
"""
from __future__ import annotations

import copy

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

import restoration as R

BRANCHES = ("identity", "A_salt", "A_blur", "A_occ")
CLASS_NAMES = ("clean", "salt_pepper", "gaussian_blur", "occlusion")


class SoftMoE(nn.Module):
    def __init__(self, gate: nn.Module, experts: list[nn.Module], tau: float = 1.0):
        super().__init__()
        assert len(experts) == 3
        self.gate = gate
        self.experts = nn.ModuleList(experts)
        self.register_buffer("tau", torch.tensor(float(tau)))

    @classmethod
    def from_task2(cls, classifier: nn.Module, specialists: dict, tau: float = 1.0):
        """Deep-copies the trained Task 2 classifier (-> gate) and specialists 1,2,3 (-> experts)."""
        return cls(copy.deepcopy(classifier), [copy.deepcopy(specialists[k]) for k in (1, 2, 3)], tau)

    def forward(self, x):
        logits = self.gate(x)
        w = torch.softmax(logits.float() / self.tau, dim=1)                     # (B, 4)
        branches = torch.stack([x] + [e(x) for e in self.experts], dim=1)      # (B, 4, 3, H, W)
        out = (w[:, :, None, None, None].to(branches.dtype) * branches).sum(1)
        return out, w, logits

    # --- training phases --------------------------------------------------------------
    def freeze_experts(self, frozen: bool = True):
        for p in self.experts.parameters():
            p.requires_grad_(not frozen)

    def train_phase(self, experts_frozen: bool):
        """Gate always trains; frozen experts are put in eval mode (no code dropout)."""
        self.train()
        if experts_frozen:
            self.experts.eval()


class MoEExport(nn.Module):
    """ONNX wrapper: input -> (output image, routing weights, gate logits)."""

    def __init__(self, moe: SoftMoE):
        super().__init__()
        self.moe = moe

    def forward(self, x):
        out, w, logits = self.moe(x)
        return out.clamp(0, 1), w, logits


def balance_loss(w: torch.Tensor) -> torch.Tensor:
    return ((w.mean(0) - 1.0 / w.shape[1]) ** 2).sum()


def moe_loss(out, target, w, logits, labels, tau, l1=0.8, ls=0.2, lc=0.1, lb=0.01):
    out, target = out.float(), target.float()
    L1 = F.l1_loss(out, target)
    ssim = R.ssim_map(out, target).mean()
    # CE on the routing distribution itself (logits / tau), so the classification term shapes the actual weights
    ce = F.cross_entropy(logits.float() / tau, labels)
    bal = balance_loss(w)
    total = l1 * L1 + ls * (1 - ssim) + lc * ce + lb * bal
    return total, {"l1": L1.detach(), "ssim": ssim.detach(), "ce": ce.detach(), "balance": bal.detach()}


def train_one_epoch(moe, loader, optimizer, scheduler, device, scaler, lam, experts_frozen, amp=True, grad_clip=1.0):
    moe.train_phase(experts_frozen)
    use_amp = amp and device.type == "cuda"
    agg = {k: 0.0 for k in ("loss", "l1", "ssim", "ce", "balance", "gate_acc")}
    n = 0
    for x, y, lab in loader:
        x, y, lab = x.to(device, non_blocking=True), y.to(device, non_blocking=True), lab.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            out, w, logits = moe(x)
        loss, parts = moe_loss(out, y, w, logits, lab, moe.tau, **lam)
        if not torch.isfinite(loss):
            raise FloatingPointError("non-finite MoE loss")
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        if grad_clip:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_([p for p in moe.parameters() if p.requires_grad], grad_clip)
        scaler.step(optimizer)
        scaler.update()
        if scheduler is not None:
            scheduler.step()
        b = len(lab)
        agg["loss"] += loss.item() * b
        for k in ("l1", "ssim", "ce", "balance"):
            agg[k] += parts[k].item() * b
        agg["gate_acc"] += (w.argmax(1) == lab).float().sum().item()
        n += b
    return {k: v / max(n, 1) for k, v in agg.items()}


@torch.no_grad()
def evaluate(moe, loader, device, amp=True, per_image=False):
    """Reconstruction metrics overall/per class, mean routing weights per true class, gate accuracy."""
    moe.eval()
    use_amp = amp and device.type == "cuda"
    P, S, M, W, L = [], [], [], [], []
    for batch in loader:
        x, y, lab = batch[0].to(device), batch[1].to(device), batch[2]
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            out, w, _ = moe(x)
        out = out.float().clamp(0, 1)
        P.append(R.psnr_per_image(out, y).cpu()); S.append(R.ssim_per_image(out, y).cpu()); M.append(R.mae_per_image(out, y).cpu())
        W.append(w.float().cpu()); L.append(lab)
    P, S, M, W, L = (torch.cat(t).numpy() for t in (P, S, M, W, L))
    res = {"psnr": float(P.mean()), "ssim": float(S.mean()), "mae": float(M.mean()),
           "gate_acc": float((W.argmax(1) == L).mean()), "weight_matrix": np.zeros((4, 4))}
    for c, n in enumerate(CLASS_NAMES):
        m = L == c
        if m.any():
            res[f"psnr/{n}"], res[f"ssim/{n}"], res[f"mae/{n}"] = float(P[m].mean()), float(S[m].mean()), float(M[m].mean())
            res["weight_matrix"][c] = W[m].mean(0)
    res["score"] = balanced_score(res)
    res["branch_usage"] = W.mean(0)                    # average weight per branch over all inputs
    if per_image:
        res["per_image"] = {"psnr": P, "ssim": S, "mae": M, "w": W, "label": L}
    return res


def balanced_score(res) -> float:
    """Mean over the 4 conditions of 0.5*SSIM + 0.5*min(PSNR/40, 1): every condition counts equally, and near-perfect
    clean reconstructions (very high PSNR) cannot dominate the objective."""
    vals = [R.val_score(res[f"psnr/{n}"], res[f"ssim/{n}"]) for n in CLASS_NAMES if f"psnr/{n}" in res]
    return float(np.mean(vals))


def collapse_check(res, min_usage=0.03, min_gate_acc=0.4) -> str | None:
    """Routing collapse = a branch that is (almost) never used, or a gate that no longer tracks the corruption."""
    u = res["branch_usage"]
    if u.min() < min_usage:
        return f"branch {BRANCHES[int(u.argmin())]} inactive (mean weight {u.min():.3f})"
    if res["gate_acc"] < min_gate_acc:
        return f"gate accuracy {res['gate_acc']:.2f} < {min_gate_acc}"
    return None

In [ ]:
import os, sys, json, time, math, shutil, importlib, warnings, gc, copy
from pathlib import Path
import numpy as np, pandas as pd, torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import optuna

sys.path.insert(0, str(Path.cwd()))
import pet_data as pdt, restoration as R, classifier as C, moe as M
for m_ in (pdt, R, C, M):
    importlib.reload(m_)
warnings.filterwarnings("ignore", category=optuna.exceptions.ExperimentalWarning)
warnings.filterwarnings("ignore", message=".*lr_scheduler.step.*")        # harmless: AMP skips a step at the start
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ============================ MAIN SWITCH ============================
MODE = "smoke"          # "smoke" first, then "full"
# =====================================================================
MODE = os.environ.get("T3_MODE", MODE)

SETTINGS = {
    "smoke": dict(train_subset=512, val_per_class=32, test_images=40, batch_size=16,
                  n_trials=3, trial_warm=1, trial_joint=1, optuna_timeout=None,
                  final_warm=1, final_joint=2, patience=99, compound_per_pair=8),
    "full":  dict(train_subset=None, val_per_class=None, test_images=None, batch_size=32,
                  n_trials=15, trial_warm=1, trial_joint=4, optuna_timeout=int(1.5 * 3600),
                  final_warm=3, final_joint=25, patience=8, compound_per_pair=100),
}
S = dict(SETTINGS[MODE])
S.update(json.loads(os.environ.get("T3_SETTINGS_OVERRIDE", "{}")))   # local CPU testing only

IS_KAGGLE  = Path("/kaggle/input").exists()
INPUT_ROOT = Path(os.environ.get("PET_INPUT_ROOT", "/kaggle/input"))
WORK_DIR   = Path(os.environ.get("WORK_DIR", "/kaggle/working" if IS_KAGGLE else "./work")).resolve()
IMAGES_DIR, ANNOT_DIR = None, None

TAG = "task3" if MODE == "full" else "task3_smoke"
DIRS = {k: WORK_DIR / k / TAG for k in ["checkpoints", "models", "figures", "results", "optuna"]}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

SEED = 42
AMP = True
NUM_WORKERS = pdt.default_num_workers()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pdt.seed_everything(SEED)
WARM_LR = 5e-4          # gate-only warm-up learning rate (fixed; joint lr is tuned and searched below it)
WEIGHT_DECAY = 1e-5
BR = list(M.BRANCHES); CN = list(M.CLASS_NAMES)

USE_WANDB = True
WANDB_PROJECT = "genai-assignment1"
WANDB_ENTITY = None
WANDB_SECRET_NAME = "WANDB_API_KEY"
GROUP = "task3" if MODE == "full" else "task3-smoke"

print(f"MODE={MODE} | device={device} {torch.cuda.get_device_name(0) if device.type == 'cuda' else ''} | "
      f"torch {torch.__version__} | optuna {optuna.__version__} | workers={NUM_WORKERS}")
print(json.dumps(S))

In [ ]:
WANDB_OK = False
if USE_WANDB:
    try:
        import wandb
        if IS_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            wandb.login(key=UserSecretsClient().get_secret(WANDB_SECRET_NAME))
        WANDB_OK = True
    except Exception as e:
        print("W&B login error:", repr(e))

def wb_init(name, job_type, config=None, **kw):
    if not WANDB_OK:
        return None
    if wandb.run is not None:
        wandb.finish()
    return wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, group=GROUP, name=name, job_type=job_type,
                      tags=["task3", MODE], config=config or {}, **kw)

def wb_log_figs(run, figs: dict):
    if run is not None:
        run.log({k: wandb.Image(str(v)) for k, v in figs.items()})

print("W&B:", "on" if WANDB_OK else "off")
if USE_WANDB and not WANDB_OK:
    raise RuntimeError("W&B login failed — attach the WANDB_API_KEY secret to THIS notebook and turn Internet on. Nothing has been trained.")

## 1. Data, Task 2 components, pre-flight gate

In [ ]:
data = pdt.prepare_data(INPUT_ROOT, WORK_DIR, IMAGES_DIR, ANNOT_DIR)
train_images = data.train_images if S["train_subset"] is None else data.train_images[:S["train_subset"]]
val_records = data.val_manifest["records"]
if S["val_per_class"] is not None:
    val_records = [r for c in range(4) for r in [x for x in val_records if x["label"] == c][:S["val_per_class"]]]
val_loader = pdt.make_eval_loader(data.val_images, data.val_name_to_row, val_records, batch_size=64, num_workers=NUM_WORKERS)

def find_ckpt(fname, parent):
    cands = sorted(INPUT_ROOT.rglob(fname))
    full = [p for p in cands if p.parent.name == parent]
    smoke = [p for p in cands if p.parent.name == f"{parent}_smoke"]
    return (full or smoke or [None])[0]

T2 = {"classifier": find_ckpt("task2_classifier_best.pth", "task2"),
      **{k: find_ckpt(f"task2_expert_{n}_best.pth", "task2") for k, n in ((1, "salt_pepper"), (2, "gaussian_blur"), (3, "occlusion"))}}
T1_CKPT = find_ckpt("task1_best.pth", "task1")
for k, v in T2.items():
    print(f"Task 2 {k}: {v}")
print("Task 1:", T1_CKPT)

def load_task2():
    ck = torch.load(T2["classifier"], map_location=device, weights_only=False)
    clf = C.CorruptionClassifier(**ck["model_config"]).to(device); clf.load_state_dict(ck["model"]); clf.eval()
    specs = {}
    for k in (1, 2, 3):
        ck = torch.load(T2[k], map_location=device, weights_only=False)
        specs[k] = R.ConvAutoencoder(**ck["model_config"]).to(device); specs[k].load_state_dict(ck["model"]); specs[k].eval()
    return clf, specs

In [ ]:
pre = []
def pf(name, ok, detail="", fatal=True):
    pre.append({"check": name, "passed": bool(ok), "fatal": fatal, "detail": str(detail)})
    print(("PASS " if ok else ("FAIL " if fatal else "WARN ")) + name + (f"  — {detail}" if detail else ""))

gpu = torch.cuda.is_available()
pf("GPU available", gpu, torch.cuda.get_device_name(0) if gpu else "Settings -> Accelerator -> GPU T4", fatal=(MODE == "full"))
wb_ok, wb_detail = False, "USE_WANDB is False"
if USE_WANDB:
    try:
        r = wb_init("preflight", "preflight", {"mode": MODE, "settings": S})
        r.log({"preflight/ok": 1}); wb_detail = r.url; r.finish(); wb_ok = True
    except Exception as e:
        wb_detail = repr(e)
pf("W&B login + test run", wb_ok or not USE_WANDB, wb_detail)
pf("dataset found", len(data.split["train"]) + len(data.split["val"]) >= 3600 and len(data.test_names) >= 3600,
   f"{len(data.split['train'])} train / {len(data.split['val'])} val / {len(data.test_names)} test")
pf("notebook-00 output attached (same split + manifests as Tasks 1-2)", data.source.startswith("attached"), data.source, fatal=False)
found = all(v is not None for v in T2.values())
pf("Task 2 classifier + 3 specialists found", found, "attach the Task 2 notebook output (Add Input -> Your Work -> Notebooks)")
if found:
    is_full = all(v.parent.name == "task2" for v in T2.values())
    pf("Task 2 components come from the FULL run", is_full, "smoke-mode checkpoints found" if not is_full else "", fatal=(MODE == "full"))
    try:
        clf0, spec0 = load_task2()
        pf("Task 2 components load", True, f"gate {clf0.config['channels']}, experts code {spec0[1].code_dim}")
    except Exception as e:
        pf("Task 2 components load", False, repr(e)[:200])
universal = None
if T1_CKPT is not None:
    try:
        ck1 = torch.load(T1_CKPT, map_location=device, weights_only=False)
        universal = R.ConvAutoencoder(**ck1["model_config"]).to(device); universal.load_state_dict(ck1["model"]); universal.eval()
    except Exception as e:
        universal = None; print("   Task 1 checkpoint not loadable:", repr(e)[:120])
pf("Task 1 v2 model attached (universal comparison)", universal is not None, "optional", fatal=False)
try:
    import onnx, onnxruntime, sklearn
    pf("onnx + onnxruntime + sklearn importable", True, f"onnx {onnx.__version__}, ort {onnxruntime.__version__}")
except Exception as e:
    pf("onnx + onnxruntime + sklearn importable", False, repr(e))
ok = True
for d in DIRS.values():
    try:
        (d / ".probe").write_text("x"); (d / ".probe").unlink()
    except Exception:
        ok = False
pf("output folders writeable", ok, str(WORK_DIR))
free = shutil.disk_usage(WORK_DIR).free / 1e9
pf("free disk > 5 GB", free > 5, f"{free:.1f} GB")
pre_df = pd.DataFrame(pre)
bad = pre_df[pre_df.fatal & ~pre_df.passed]
assert bad.empty, "PRE-FLIGHT FAILED — nothing has been trained. Fix these and re-run:\n" + bad.to_string()
print(f"\nPRE-FLIGHT OK ({int(pre_df.passed.sum())}/{len(pre_df)} passed) — safe to train.")

In [ ]:
def records_to_arrays(records, images, name_to_row):
    clean = np.stack([images[name_to_row[r["image"]]].astype(np.float32) / 255 for r in records])
    corrupted = np.stack([pdt.apply_corruption(c, r) for c, r in zip(clean, records)])
    return corrupted, clean

def build_moe(tau):
    # fresh deep copies of the frozen Task 2 components every time -> every trial starts from the same point
    return M.SoftMoE.from_task2(clf0, spec0, tau).to(device)

def wflat(res, prefix="val"):
    # 4x4 routing matrix (true condition x branch) -> flat dict for W&B
    return {f"{prefix}/w/{CN[c]}/{BR[k]}": float(res["weight_matrix"][c, k]) for c in range(4) for k in range(4)}

## 2. Unit tests

In [ ]:
tests = []
def check(name, ok, detail=""):
    tests.append({"test": name, "passed": bool(ok), "detail": str(detail)})
    print(("PASS " if ok else "FAIL ") + name + (f"  — {detail}" if detail else ""))

vb = next(iter(val_loader)); x, y, lab = vb[0][:8].to(device), vb[1][:8].to(device), vb[2][:8].to(device)
moe_t = build_moe(1.0).eval()
with torch.no_grad():
    out, w, lg = moe_t(x)
    br = [x] + [e(x) for e in moe_t.experts]
    ref = sum(w[:, k, None, None, None] * br[k] for k in range(4))
check("weights are a distribution (sum to 1, >= 0)", torch.allclose(w.sum(1), torch.ones(len(w), device=device), atol=1e-5) and (w >= 0).all())
check("output = w0*x + w1*A_salt + w2*A_blur + w3*A_occ", torch.allclose(out.float(), ref.float(), atol=1e-4))
with torch.no_grad():
    check("gate initialised from Task 2 classifier (identical logits)", torch.allclose(lg, clf0(x), atol=1e-5))
    check("experts initialised from Task 2 specialists (identical outputs)", all(torch.allclose(moe_t.experts[k - 1](x), spec0[k](x), atol=1e-5) for k in (1, 2, 3)))
    moe_t.tau.fill_(1e-4); out_h, _, lg = moe_t(x); a = lg.argmax(1)
    hard = torch.stack([br[int(a[i])][i] for i in range(len(a))])
check("tau -> 0 reproduces Task 2 hard routing exactly", torch.allclose(out_h, hard, atol=1e-3))
moe_t.tau.fill_(1.0); moe_t.freeze_experts(True); moe_t.train_phase(True); moe_t.zero_grad()
out, w, lg = moe_t(x); loss, parts = M.moe_loss(out, y, w, lg, lab, moe_t.tau); loss.backward()
check("warm-up: experts frozen (no grads, eval mode), gate trains",
      all(p.grad is None for p in moe_t.experts.parameters()) and all(p.grad is not None for p in moe_t.gate.parameters()) and not moe_t.experts.training)
moe_t.freeze_experts(False); moe_t.train_phase(False); moe_t.zero_grad()
out, w, lg = moe_t(x); M.moe_loss(out, y, w, lg, lab, moe_t.tau)[0].backward()
check("joint phase: gradients reach gate and all experts", all(p.grad is not None for p in moe_t.parameters()))
check("balance loss: 0 for uniform routing, 0.75 for total collapse",
      abs(M.balance_loss(torch.full((8, 4), 0.25)).item()) < 1e-9 and abs(M.balance_loss(torch.eye(4)[[0] * 8]).item() - 0.75) < 1e-6)
bl = pdt.make_train_loader(train_images, batch_size=S["batch_size"], balanced=True, num_workers=0)
cnt = np.bincount(next(iter(bl))[2].numpy(), minlength=4)
check("training batches exactly balanced", (cnt == S["batch_size"] // 4).all(), cnt.tolist())
del moe_t, bl
tests_df = pd.DataFrame(tests)
assert tests_df.passed.all(), f"UNIT TESTS FAILED:\n{tests_df[~tests_df.passed]}"
print(f"\nall {len(tests_df)} tests passed")

## 3. Reference point: the initialised system before any joint training
The gate is the Task 2 classifier at τ = 1, and the experts are the Task 2 specialists.

In [ ]:
ev0 = M.evaluate(build_moe(1.0), val_loader, device, AMP)
print(f"initialised soft MoE (tau=1): score {ev0['score']:.4f} | PSNR {ev0['psnr']:.2f} SSIM {ev0['ssim']:.4f} | gate acc {ev0['gate_acc']:.3f}")
print(pd.DataFrame(ev0["weight_matrix"], index=CN, columns=BR).round(3))

## 4. Optuna search over joint fine-tuning
* **Tuned:** joint learning rate, temperature τ, classification weight λ_c, balance weight λ_b, and the reconstruction weighting (λ₁ = α, λ_s = 1 − α).
* **Objective:** the mean over the four conditions of 0.5·SSIM + 0.5·min(PSNR/40, 1) on the validation manifest. Each condition counts equally, and near-perfect clean reconstructions cannot dominate.
* **Pruning:** the median pruner, plus a **routing-collapse** rule: a trial is stopped as soon as any branch's mean weight falls below 0.03 or the gate's accuracy falls below 0.4.
* The first trial is the suggested starting point (λ₁ = 0.8, λ_s = 0.2, λ_c = 0.1, λ_b = 0.01, τ = 1).

In [ ]:
SPACE = {
    "joint_lr": {"type": "float", "low": 1e-5, "high": 3e-4, "log": True},
    "tau":      {"type": "float", "low": 0.5, "high": 3.0, "log": True},
    "lambda_c": {"type": "float", "low": 0.01, "high": 1.0, "log": True},
    "lambda_b": {"type": "float", "low": 1e-3, "high": 0.3, "log": True},
    "alpha":    {"type": "float", "low": 0.5, "high": 0.95},          # lambda_1 = alpha, lambda_s = 1 - alpha
}
BASELINE = {"joint_lr": 1e-4, "tau": 1.0, "lambda_c": 0.1, "lambda_b": 0.01, "alpha": 0.8}
pdt.save_json(SPACE, DIRS["results"] / "search_space.json", indent=1)

def lam_of(hp):
    return {"l1": hp["alpha"], "ls": 1 - hp["alpha"], "lc": hp["lambda_c"], "lb": hp["lambda_b"]}

def run_training(hp, warm_epochs, joint_epochs, run=None, trial=None, ckpt=None, patience=None, verbose=False):
    torch.manual_seed(SEED + (trial.number if trial else 0))
    moe = build_moe(hp["tau"])
    loader = pdt.make_train_loader(train_images, batch_size=S["batch_size"], balanced=True, num_workers=NUM_WORKERS)
    lam, hist = lam_of(hp), []
    best, best_ep, step = -1.0, -1, 0
    try:
        for phase, E in (("warmup", warm_epochs), ("joint", joint_epochs)):
            if E == 0:
                continue
            frozen = phase == "warmup"
            moe.freeze_experts(frozen)
            params = [p for p in moe.parameters() if p.requires_grad]
            lr = WARM_LR if frozen else hp["joint_lr"]
            opt = torch.optim.AdamW(params, lr=lr, weight_decay=WEIGHT_DECAY)
            total = max(1, len(loader) * E)
            sch = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: 0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * min(1.0, s / total))))
            scaler = R.make_scaler(device, AMP)
            for ep in range(E):
                t0 = time.time()
                tr = M.train_one_epoch(moe, loader, opt, sch, device, scaler, lam, frozen, AMP)
                ev = M.evaluate(moe, val_loader, device, AMP)
                rec = {"step": step, "phase": phase, "epoch_in_phase": ep, "lr": sch.get_last_lr()[0], "time_s": time.time() - t0,
                       **{f"train/{k}": v for k, v in tr.items()}, "val/score": ev["score"], "val/psnr": ev["psnr"], "val/ssim": ev["ssim"],
                       "val/gate_acc": ev["gate_acc"], **{f"val/{k}": v for k, v in ev.items() if "/" in k},
                       **{f"val/usage/{BR[k]}": float(ev["branch_usage"][k]) for k in range(4)}, **wflat(ev)}
                hist.append(rec)
                if run:
                    run.log({**{k: v for k, v in rec.items() if k != "phase"}, "phase_joint": int(phase == "joint")})
                improved = ev["score"] > best
                if improved:
                    best, best_ep = ev["score"], step
                    if trial:
                        for k in ("psnr", "ssim", "gate_acc"):
                            trial.set_user_attr(f"val_{k}", ev[k])
                    if ckpt:
                        torch.save({"model": moe.state_dict(), "hp": hp, "step": step, "phase": phase, "tau": hp["tau"],
                                    "gate_config": moe.gate.config, "expert_configs": [e.config for e in moe.experts],
                                    "val": {k: (v.tolist() if isinstance(v, np.ndarray) else v) for k, v in ev.items()}}, ckpt)
                if verbose:
                    print(f"[{phase:6s}] step {step:3d} | loss {tr['loss']:.4f} (L1 {tr['l1']:.4f} SSIM {tr['ssim']:.3f} CE {tr['ce']:.3f} "
                          f"bal {tr['balance']:.4f}) | val score {ev['score']:.4f} PSNR {ev['psnr']:.2f} gate acc {ev['gate_acc']:.3f}"
                          f"{' *' if improved else ''} | {rec['time_s']:.0f}s")
                if trial:
                    trial.report(ev["score"], step)
                    why = M.collapse_check(ev)
                    if why:
                        trial.set_user_attr("collapse", why); raise optuna.TrialPruned(why)
                    if trial.should_prune():
                        raise optuna.TrialPruned()
                step += 1
                if patience and phase == "joint" and step - 1 - best_ep >= patience:
                    print(f"early stopping (no improvement for {patience} epochs)"); break
    finally:
        del loader
    return moe, best, hist

def objective(trial):
    hp = {k: trial.suggest_float(k, s["low"], s["high"], log=s.get("log", False)) for k, s in SPACE.items()}
    run = wb_init(f"moe-trial-{trial.number:03d}", "moe-trial", {**hp, "trial": trial.number, "warm": S["trial_warm"], "joint": S["trial_joint"]})
    state = "complete"
    try:
        _, best, _ = run_training(hp, S["trial_warm"], S["trial_joint"], run, trial)
    except optuna.TrialPruned:
        state = trial.user_attrs.get("collapse") and "collapse" or "pruned"; raise
    except torch.cuda.OutOfMemoryError:
        state = "oom"; raise optuna.TrialPruned("OOM")
    finally:
        if run:
            run.summary.update({"state": state}); run.finish()
        gc.collect(); torch.cuda.empty_cache()
    return best

In [ ]:
STUDY = f"{TAG}_soft_moe"
DB = DIRS["optuna"] / f"{STUDY}.db"
if MODE == "smoke" and DB.exists():
    DB.unlink()
study = optuna.create_study(study_name=STUDY, storage=f"sqlite:///{DB}", direction="maximize", load_if_exists=True,
                            sampler=optuna.samplers.TPESampler(seed=SEED, multivariate=True, n_startup_trials=min(6, S["n_trials"])),
                            pruner=optuna.pruners.MedianPruner(n_startup_trials=4, n_warmup_steps=S["trial_warm"] + 1))
if len(study.trials) == 0:
    study.enqueue_trial(BASELINE)
t0 = time.time()
done = lambda: len([t for t in study.trials if t.state in (optuna.trial.TrialState.COMPLETE, optuna.trial.TrialState.PRUNED)])
while done() < S["n_trials"]:
    left = None if S["optuna_timeout"] is None else S["optuna_timeout"] - (time.time() - t0)
    if left is not None and left <= 0:
        print("time budget reached"); break
    study.optimize(objective, n_trials=1, timeout=left, gc_after_trial=True, catch=(FloatingPointError,))
print(f"study time {(time.time() - t0) / 60:.1f} min")

tdf = study.trials_dataframe(attrs=("number", "value", "state", "params", "user_attrs", "duration"))
tdf.to_csv(DIRS["results"] / "optuna_trials.csv", index=False)
states = tdf["state"].value_counts().to_dict()
if "user_attrs_collapse" in tdf:
    states["COLLAPSE (pruned)"] = int(tdf["user_attrs_collapse"].notna().sum())
completed = [t_ for t_ in study.trials if t_.state == optuna.trial.TrialState.COMPLETE]
if completed:
    BEST_HP, BEST_NUM, BEST_VAL = dict(study.best_params), study.best_trial.number, study.best_value
else:        # every trial pruned / collapsed -> fall back to the suggested starting configuration
    BEST_HP, BEST_NUM, BEST_VAL = dict(BASELINE), None, None
    print("WARNING: no trial completed (all pruned or collapsed) — using the baseline configuration. Discuss in the report.")
print("trial states:", states)
print(f"best trial #{BEST_NUM}: {BEST_VAL}", json.dumps(BEST_HP, indent=1))
display(tdf.sort_values("value", ascending=False).head(8))
figs = {}
from optuna.visualization import matplotlib as ovm
for nm, fn in ([("history", ovm.plot_optimization_history), ("importance", ovm.plot_param_importances),
               ("slice", ovm.plot_slice), ("parallel", ovm.plot_parallel_coordinate)] if completed else []):
    try:
        obj = fn(study); fig = (obj.flat[0] if hasattr(obj, "flat") else obj).figure
        fig.set_size_inches(12 if nm in ("slice", "parallel") else 7, 4.5)
        p = DIRS["figures"] / f"optuna_{nm}.png"; fig.savefig(p, dpi=140, bbox_inches="tight"); plt.show(); figs[f"optuna/{nm}"] = p
    except Exception as e:
        print(f"plot {nm} skipped: {e}")
plt.style.use("default")
pdt.save_json({"best_trial": BEST_NUM, "best_value": BEST_VAL, "best_params": BEST_HP,
               "user_attrs": study.best_trial.user_attrs if completed else {}, "n_trials": len(study.trials), "states": states,
               "search_space": SPACE, "baseline": BASELINE, "warm_lr": WARM_LR, "batch_size": S["batch_size"]},
              DIRS["results"] / "optuna_best.json", indent=1)
run = wb_init("moe-optuna-summary", "optuna-summary", {"search_space": SPACE})
if run:
    run.summary.update({"best_value": BEST_VAL, "best_trial": BEST_NUM, "states": states,
                        **{f"best/{k}": v for k, v in BEST_HP.items()}})
    run.log({"optuna_trials": wandb.Table(dataframe=tdf.astype(str))}); wb_log_figs(run, figs)
    art = wandb.Artifact("task3-soft-moe-study", type="optuna-study"); art.add_file(str(DB)); run.log_artifact(art); run.finish()

## 5. Final training: warm-up (experts frozen) → joint fine-tuning

In [ ]:
CKPT = DIRS["checkpoints"] / "task3_soft_moe_best.pth"
run = wb_init("moe-final", "moe-final", {**BEST_HP, "warm_epochs": S["final_warm"], "joint_epochs": S["final_joint"],
                                         "patience": S["patience"], "warm_lr": WARM_LR, "batch_size": S["batch_size"]})
t0 = time.time()
moe, best, hist = run_training(BEST_HP, S["final_warm"], S["final_joint"], run, None, CKPT, S["patience"], verbose=True)
ck = torch.load(CKPT, map_location=device, weights_only=False)
moe.load_state_dict(ck["model"]); moe.eval()
print(f"training {(time.time() - t0) / 60:.1f} min | best step {ck['step']} ({ck['phase']}) | val score {ck['val']['score']:.4f}")

h = pd.DataFrame(hist); h.to_csv(DIRS["results"] / "training_history.csv", index=False)
fig, ax = plt.subplots(1, 4, figsize=(19, 3.6))
ax[0].plot(h.step, h["train/loss"], label="total"); ax[0].plot(h.step, h["train/l1"], label="L1"); ax[0].plot(h.step, 1 - h["train/ssim"], label="1-SSIM")
ax[0].plot(h.step, h["train/ce"], label="CE"); ax[0].plot(h.step, h["train/balance"], label="balance"); ax[0].set_yscale("log"); ax[0].legend(fontsize=7); ax[0].set_title("training loss terms")
for n in CN:
    ax[1].plot(h.step, h[f"val/psnr/{n}"], label=n)
ax[1].set_title("val PSNR per condition"); ax[1].legend(fontsize=7)
ax[2].plot(h.step, h["val/score"], "k", label="val score"); ax[2].plot(h.step, h["val/gate_acc"], "C3", label="gate accuracy"); ax[2].legend(fontsize=7); ax[2].set_title("objective / gate accuracy")
for c in range(4):
    ax[3].plot(h.step, h[f"val/w/{CN[c]}/{BR[c]}"], label=f"w_{c} on {CN[c]}")
ax[3].set_title("weight on the correct branch (val)"); ax[3].legend(fontsize=7); ax[3].set_ylim(0, 1.02)
warm_end = h[h.phase == "warmup"].step.max() if (h.phase == "warmup").any() else None
for a in ax:
    a.set_xlabel("epoch"); a.axvline(ck["step"], color="grey", ls=":")
    if warm_end is not None:
        a.axvspan(-0.5, warm_end + 0.5, color="C0", alpha=0.08)
fig.suptitle("Soft MoE training (shaded = warm-up with frozen experts)"); fig.tight_layout()
fig.savefig(DIRS["figures"] / "training_curves.png", dpi=150); plt.show()
if run:
    wb_log_figs(run, {"figures/training_curves": DIRS["figures"] / "training_curves.png"})
    art = wandb.Artifact("task3-soft-moe-checkpoint", type="model", metadata=BEST_HP); art.add_file(str(CKPT)); run.log_artifact(art)

## 6. Test set: soft MoE vs hard routing vs universal
Every test record is scored under five systems: the corrupted **input**, the Task 2 **hard routing** (predicted and oracle), the **soft MoE**, and the Task 1 **universal** model (if attached).
An identity output on a clean input is exact (PSNR = ∞), so those cells are left blank. The soft MoE never copies exactly (w₀ < 1), so its clean PSNR is finite.

In [ ]:
test_images, test_row, _ = data.load_test_images()
test_records = [r for r in data.test_manifest["records"] if r["image"] in test_row]
if S["test_images"] is not None:
    keep = set(data.test_names[:S["test_images"]]); test_records = [r for r in test_records if r["image"] in keep]
test_loader = pdt.make_eval_loader(test_images, test_row, test_records, batch_size=64, return_index=True, num_workers=NUM_WORKERS)

@torch.no_grad()
def eval_systems(batches, N, have_labels=True):
    # batches yield (x, y, label, idx); returns per-record metrics for every system + weights
    keys = ["in", "soft", "hard"] + (["oracle"] if have_labels else []) + (["univ"] if universal is not None else [])
    out = {f"{s}_{m}": np.zeros(N, np.float32) for s in keys for m in ("psnr", "ssim", "mae")}
    out.update({"w": np.zeros((N, 4), np.float32), "gate_pred": np.zeros(N, int), "clf_pred": np.zeros(N, int), "label": np.zeros(N, int)})
    use_amp = AMP and device.type == "cuda"
    for x, y, lab, idx in batches:
        x, y = x.to(device), y.to(device); idx = np.asarray(idx); B = len(idx); ar = torch.arange(B, device=device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            soft, w, _ = moe(x)
            pred0 = clf0(x).argmax(1)
            br = torch.stack([x] + [spec0[k](x) for k in (1, 2, 3)], 1)
            u = universal(x) if universal is not None else None
        systems = {"in": x, "soft": soft, "hard": br[ar, pred0]}
        if have_labels:
            systems["oracle"] = br[ar, lab.to(device)]
        if u is not None:
            systems["univ"] = u
        for s, o in systems.items():
            o = o.float().clamp(0, 1)
            out[f"{s}_psnr"][idx] = R.psnr_per_image(o, y).cpu().numpy()
            out[f"{s}_ssim"][idx] = R.ssim_per_image(o, y).cpu().numpy()
            out[f"{s}_mae"][idx] = R.mae_per_image(o, y).cpu().numpy()
        out["w"][idx] = w.float().cpu().numpy(); out["gate_pred"][idx] = w.argmax(1).cpu().numpy()
        out["clf_pred"][idx] = pred0.cpu().numpy(); out["label"][idx] = lab.numpy()
    for s in keys:                      # exact identity copies -> infinite PSNR -> NaN (never averaged)
        v = out[f"{s}_psnr"]; v[v >= 99.9] = np.nan
    return out

t0 = time.time()
E = eval_systems(test_loader, len(test_records))
print(f"{len(test_records)} test records scored under {'5' if universal is not None else '4'} systems in {time.time() - t0:.0f}s")
df = pd.DataFrame({"id": [r["id"] for r in test_records], "image": [r["image"] for r in test_records],
                   "type": [r["type"] for r in test_records], "severity": [r["severity_level"] for r in test_records],
                   **{k: v for k, v in E.items() if k != "w"}, **{f"w_{k}": E["w"][:, k] for k in range(4)}})
df["w_max"] = E["w"].max(1)
df["entropy"] = -(E["w"] * np.log(np.clip(E["w"], 1e-9, 1))).sum(1)
df.to_csv(DIRS["results"] / "test_per_record.csv", index=False)

order = CN
SYS = ["in", "hard", "oracle", "soft"] + (["univ"] if universal is not None else [])
NAME = {"in": "input", "hard": "hard (pred.)", "oracle": "hard (oracle)", "soft": "soft MoE", "univ": "universal (T1)"}
def summ(g):
    row = {"n": len(g)}
    for s in SYS:
        row[f"PSNR {NAME[s]}"] = g[f"{s}_psnr"].mean(); row[f"SSIM {NAME[s]}"] = g[f"{s}_ssim"].mean()
    row["ΔSSIM soft-hard"] = (g.soft_ssim - g.hard_ssim).mean()
    row["gate acc (soft)"] = (g.gate_pred == g.label).mean(); row["clf acc (T2)"] = (g.clf_pred == g.label).mean()
    return pd.Series(row)
by_type = df.groupby("type").apply(summ).reindex(order)
by_type.loc["all corrupted"] = summ(df[df.type != "clean"])
by_sev = df.groupby(["type", "severity"]).apply(summ)
by_sev = by_sev.reindex([(t, s) for t in order for s in ["none", "low", "medium", "high"] if (t, s) in by_sev.index])
for tab_ in (by_type, by_sev):          # clean rows: identity-based systems are exact (∞) -> blank, never a partial mean
    idx_ = [i for i in tab_.index if i == "clean" or (isinstance(i, tuple) and i[0] == "clean")]
    tab_.loc[idx_, [f"PSNR {NAME[s]}" for s in ("in", "hard", "oracle")]] = np.nan
print("Test results by condition:"); display(by_type.round(3))
print("By condition x severity:"); display(by_sev.round(3))
by_type.to_csv(DIRS["results"] / "test_by_type.csv"); by_sev.to_csv(DIRS["results"] / "test_by_type_severity.csv")
for nm, t in [("test_by_type", by_type), ("test_by_type_severity", by_sev)]:
    try:
        (DIRS["results"] / f"{nm}.tex").write_text(t.round(3).to_latex(float_format="%.3f"))
    except Exception as e:
        print("LaTeX skipped:", e)

fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
g = df[df.type != "clean"].groupby(["type", "severity"])
for i, (m, lab_) in enumerate([("psnr", "PSNR (dB)"), ("ssim", "SSIM")]):
    for t, c in zip(order[1:], ["C1", "C2", "C3"]):
        mm = g[[f"{s}_{m}" for s in SYS]].mean()
        for s, st in [("in", ":x"), ("hard", "--s"), ("soft", "-o")] + ([("univ", "-.^")] if universal is not None else []):
            ax[i].plot(["low", "medium", "high"], [mm.loc[(t, sv), f"{s}_{m}"] for sv in ("low", "medium", "high")], st, color=c,
                       alpha=1 if s == "soft" else .55, label=f"{t} {NAME[s]}")
    ax[i].set_ylabel(lab_); ax[i].set_xlabel("severity")
ax[1].legend(fontsize=6, ncol=2); fig.suptitle("Soft MoE (solid) vs hard routing (dashed) vs input (dotted)")
fig.tight_layout(); fig.savefig(DIRS["figures"] / "test_severity_curves.png", dpi=150); plt.show()

## 7. Gate behaviour: routing weights, expert activity, dominant vs distributed routing

In [ ]:
# Routing heatmap: mean weight per branch for every true condition x severity
W_sev = df.groupby(["type", "severity"])[[f"w_{k}" for k in range(4)]].mean()
W_sev = W_sev.reindex([(t, s) for t in order for s in ["none", "low", "medium", "high"] if (t, s) in W_sev.index])
W_sev.columns = BR
W_sev.to_csv(DIRS["results"] / "routing_weights_by_type_severity.csv")
W_type = df.groupby("type")[[f"w_{k}" for k in range(4)]].mean().reindex(order); W_type.columns = BR
W_type.to_csv(DIRS["results"] / "routing_weights_by_type.csv")
print("Mean routing weights by true condition:"); display(W_type.round(3))
print("... and by condition x severity:"); display(W_sev.round(3))

fig, ax = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={"width_ratios": [1, 1.25], "wspace": 0.55})
for a, (mat, ttl) in zip(ax, [(W_type, "by true condition"), (W_sev, "by condition × severity")]):
    im = a.imshow(mat.values, cmap="viridis", vmin=0, vmax=1, aspect="auto")
    a.set_xticks(range(4)); a.set_xticklabels(BR); a.set_yticks(range(len(mat)))
    a.set_yticklabels([i if isinstance(i, str) else f"{i[0]} / {i[1]}" for i in mat.index], fontsize=8)
    for i in range(len(mat)):
        for j in range(4):
            a.text(j, i, f"{mat.values[i, j]:.2f}", ha="center", va="center", fontsize=8, color="white" if mat.values[i, j] < .5 else "black")
    a.set_title(f"Mean routing weight {ttl}"); a.set_xlabel("branch")
fig.colorbar(im, ax=ax, fraction=0.025); fig.savefig(DIRS["figures"] / "routing_heatmap.png", dpi=150, bbox_inches="tight"); plt.show()

# Weight distributions (not only means): one panel per true condition
fig, ax = plt.subplots(1, 4, figsize=(16, 3.4), sharey=True)
for c, a in enumerate(ax):
    sub = df[df.label == c]
    a.boxplot([sub[f"w_{k}"] for k in range(4)], showfliers=False)
    a.set_xticks(range(1, 5)); a.set_xticklabels(BR, fontsize=8); a.set_title(f"true: {CN[c]}"); a.set_ylim(-0.02, 1.02)
ax[0].set_ylabel("routing weight"); fig.suptitle("Distribution of routing weights per true condition (test)")
fig.tight_layout(); fig.savefig(DIRS["figures"] / "routing_weight_distributions.png", dpi=150); plt.show()

# Expert activity checks
usage = E["w"].mean(0); argmax_share = np.bincount(E["gate_pred"], minlength=4) / len(df)
activity = pd.DataFrame({"mean weight (all inputs)": usage, "argmax share": argmax_share,
                         "mean weight on own condition": [W_type.values[k, k] for k in range(4)],
                         "max mean weight on another condition": [max(W_type.values[c, k] for c in range(4) if c != k) for k in range(4)]}, index=BR)
activity.to_csv(DIRS["results"] / "expert_activity.csv"); display(activity.round(3))
inactive = [BR[k] for k in range(4) if usage[k] < 0.05]
dominating = [(BR[k], CN[c], W_type.values[c, k]) for k in range(4) for c in range(4) if c != k and W_type.values[c, k] > 0.3]
print("Inactive branches (mean weight < 0.05):", inactive or "none")
print("Branches dominating an UNRELATED condition (mean weight > 0.3):", [f"{b} on {c} ({v:.2f})" for b, c, v in dominating] or "none")
dominant_frac = (df.w_max > 0.9).mean(); distributed_frac = (df.w_max < 0.6).mean()
print(f"one branch dominates (max w > 0.9): {dominant_frac:.1%} of test inputs | distributed (max w < 0.6): {distributed_frac:.1%}")

In [ ]:
rec_by_id = {r["id"]: r for r in test_records}
dfi = df.set_index("id")

@torch.no_grad()
def moe_panels(recs):
    cor, cl = records_to_arrays(recs, test_images, test_row)
    xx = torch.from_numpy(cor.transpose(0, 3, 1, 2)).to(device)
    soft, w, _ = moe(xx)
    pred0 = clf0(xx).argmax(1)
    br = torch.stack([xx] + [spec0[k](xx) for k in (1, 2, 3)], 1)
    hard = br[torch.arange(len(recs), device=device), pred0]
    f = lambda t: t.float().clamp(0, 1).cpu().numpy().transpose(0, 2, 3, 1)
    return cor, cl, f(soft), f(hard), w.float().cpu().numpy(), pred0.cpu().numpy()

def moe_grid(ids, path, title, recs=None, extra=None):
    recs = recs or [rec_by_id[i] for i in ids]
    cor, cl, soft, hard, w, pred0 = moe_panels(recs)
    n = len(recs)
    fig, axes = plt.subplots(n, 5, figsize=(13, 2.5 * n), squeeze=False, gridspec_kw={"width_ratios": [1, 1, 1, 1, 1.1]})
    for r in range(n):
        lab_in = pdt.describe_params(recs[r]) if "label" in recs[r] and recs[r]["label"] >= 0 else recs[r].get("desc", "")
        ms, mh = R.ssim_per_image(torch.from_numpy(soft[r:r+1].transpose(0, 3, 1, 2)), torch.from_numpy(cl[r:r+1].transpose(0, 3, 1, 2))).item(), \
                 R.ssim_per_image(torch.from_numpy(hard[r:r+1].transpose(0, 3, 1, 2)), torch.from_numpy(cl[r:r+1].transpose(0, 3, 1, 2))).item()
        for c, (im, t) in enumerate([(cl[r], "clean target"), (cor[r], f"input: {lab_in}"),
                                     (hard[r], f"hard → {BR[pred0[r]]}\nSSIM {mh:.3f}"), (soft[r], f"soft MoE\nSSIM {ms:.3f}")]):
            axes[r, c].imshow(np.clip(im, 0, 1)); axes[r, c].set_title(t, fontsize=7); axes[r, c].set_xticks([]); axes[r, c].set_yticks([])
        a = axes[r, 4]; a.barh(range(4), w[r], color=["0.5", "C1", "C2", "C3"]); a.set_yticks(range(4)); a.set_yticklabels(BR, fontsize=7)
        a.set_xlim(0, 1); a.invert_yaxis(); a.set_title("routing weights", fontsize=7)
        for k in range(4):
            a.text(min(w[r, k] + 0.02, 0.8), k, f"{w[r, k]:.2f}", va="center", fontsize=7)
    fig.suptitle(title, fontsize=11); fig.tight_layout(); fig.savefig(path, dpi=140, bbox_inches="tight"); plt.show()
    return path

figs = {}
# Dominant routing: one confident example per condition
dom = [df[(df.label == c) & (df.w_max > 0.9)].sort_values("w_max", ascending=False).head(1).id for c in range(4)]
dom = [i for s in dom for i in s]
if dom:
    figs["figures/dominant_routing"] = moe_grid(dom, DIRS["figures"] / "examples_dominant_routing.png", "One expert dominates (max weight > 0.9)")
# Distributed routing: the least confident gate decisions (highest entropy), from different conditions where possible
dist = df.sort_values("entropy", ascending=False).drop_duplicates("type").head(4).id.tolist()
figs["figures/distributed_routing"] = moe_grid(dist, DIRS["figures"] / "examples_distributed_routing.png", "Weights distributed across several branches (highest entropy)")
# Representative: per condition x severity, closest to the median soft SSIM
rep = []
for t in order:
    for s in (["none"] if t == "clean" else ["low", "medium", "high"]):
        g_ = df[(df.type == t) & (df.severity == s)]
        if len(g_):
            rep.append(g_.iloc[(g_.soft_ssim - g_.soft_ssim.median()).abs().argsort().iloc[0]]["id"])
figs["figures/representative"] = moe_grid(rep[:10], DIRS["figures"] / "examples_representative.png", "Soft MoE — representative test examples")
# Failure cases: lowest soft SSIM (2) and where soft is most worse than hard routing (2)
fail = list(df.sort_values("soft_ssim").drop_duplicates("type").head(2).id)
fail += [i for i in df.assign(d=df.soft_ssim - df.hard_ssim).sort_values("d").id if i not in fail][:2]
figs["figures/failures"] = moe_grid(fail, DIRS["figures"] / "examples_failures.png", "Soft MoE failure cases (lowest SSIM; largest loss vs hard routing)")
dfi.loc[fail, ["type", "severity", "hard_ssim", "soft_ssim", "w_0", "w_1", "w_2", "w_3"]].to_csv(DIRS["results"] / "failure_cases.csv")

## 8. Stress test: compound corruptions
Hard routing has to choose one expert, so it cannot handle **two corruptions at once**. A soft mixture can blend branches.
Deterministic pairs of medium-severity corruptions are applied, in sequence, to test images. They are **not** seen in training. Their parameters are saved as a manifest.

In [ ]:
PAIRS = [("salt_pepper + gaussian_blur", pdt.SALT_PEPPER, pdt.BLUR), ("gaussian_blur + occlusion", pdt.BLUR, pdt.OCCLUSION),
         ("salt_pepper + occlusion", pdt.SALT_PEPPER, pdt.OCCLUSION)]
rng = np.random.default_rng(np.random.SeedSequence([SEED, 3]))
names = sorted(set(r["image"] for r in test_records))
comp = []
for pname, a, b in PAIRS:
    for nm in rng.choice(names, size=min(S["compound_per_pair"], len(names)), replace=False):
        comp.append({"image": str(nm), "pair": pname, "first": pdt.fixed_params(a, "medium", int(rng.integers(2**31 - 1))),
                     "second": pdt.fixed_params(b, "medium", int(rng.integers(2**31 - 1)))})
pdt.save_json(comp, DIRS["results"] / "compound_manifest.json")
cl_c = np.stack([test_images[test_row[c["image"]]].astype(np.float32) / 255 for c in comp])
co_c = np.stack([pdt.apply_corruption(pdt.apply_corruption(x_, c["first"]), c["second"]) for x_, c in zip(cl_c, comp)])
def comp_batches(bs=64):
    for i in range(0, len(comp), bs):
        yield (torch.from_numpy(co_c[i:i+bs].transpose(0, 3, 1, 2)), torch.from_numpy(cl_c[i:i+bs].transpose(0, 3, 1, 2)),
               torch.full((min(bs, len(comp) - i),), -1), np.arange(i, min(i + bs, len(comp))))
EC = eval_systems(comp_batches(), len(comp), have_labels=False)
cdf = pd.DataFrame({"pair": [c["pair"] for c in comp], **{k: v for k, v in EC.items() if k not in ("w", "label")},
                    **{f"w_{k}": EC["w"][:, k] for k in range(4)}})
comp_tab = cdf.groupby("pair")[[c for c in cdf.columns if c.endswith(("_psnr", "_ssim")) or c.startswith("w_")]].mean()
comp_tab["soft better than hard (SSIM)"] = cdf.assign(b=cdf.soft_ssim > cdf.hard_ssim).groupby("pair").b.mean()
comp_tab.to_csv(DIRS["results"] / "compound_results.csv")
print("Compound corruptions (medium + medium):"); display(comp_tab.round(3))
ex = [cdf[cdf.pair == p].index[0] for p, _, _ in PAIRS]
recs_c = [{"image": comp[i]["image"], "desc": comp[i]["pair"].replace("gaussian_blur", "blur").replace("salt_pepper", "s&p"), "label": -1,
           "_arr": co_c[i]} for i in ex]
# render with the compound inputs (records_to_arrays can't rebuild them) -> temporary override
_orig = records_to_arrays
records_to_arrays = lambda recs, im, row: (np.stack([r["_arr"] for r in recs]), np.stack([im[row[r["image"]]].astype(np.float32) / 255 for r in recs]))
figs["figures/compound"] = moe_grid(None, DIRS["figures"] / "examples_compound.png", "Compound corruptions: hard routing vs soft MoE", recs=recs_c)
records_to_arrays = _orig

In [ ]:
run = wb_init("moe-eval", "eval", {**BEST_HP, "universal_attached": universal is not None})
if run:
    run.summary.update({"test/all_corrupted/" + k: float(v) for k, v in by_type.loc["all corrupted"].items() if pd.notna(v)})
    run.summary.update({"routing/dominant_frac": float(dominant_frac), "routing/distributed_frac": float(distributed_frac),
                        "routing/inactive": ", ".join(inactive) or "none", **{f"routing/usage/{BR[k]}": float(usage[k]) for k in range(4)}})
    run.log({"test/by_type": wandb.Table(dataframe=by_type.reset_index().rename(columns={"index": "type"})),
             "test/by_type_severity": wandb.Table(dataframe=by_sev.reset_index()),
             "routing/weights_by_type": wandb.Table(dataframe=W_type.reset_index()),
             "routing/weights_by_type_severity": wandb.Table(dataframe=W_sev.reset_index()),
             "routing/expert_activity": wandb.Table(dataframe=activity.reset_index()),
             "compound/results": wandb.Table(dataframe=comp_tab.reset_index())})
    wb_log_figs(run, {**figs, "figures/routing_heatmap": DIRS["figures"] / "routing_heatmap.png",
                      "figures/routing_weight_distributions": DIRS["figures"] / "routing_weight_distributions.png",
                      "figures/test_severity_curves": DIRS["figures"] / "test_severity_curves.png"})

## 9. ONNX export of the complete soft-MoE pipeline
One graph: input image → (restored image, 4 routing weights, gate logits). The temperature τ is built in.

In [ ]:
ONNX_PATH = DIRS["models"] / "task3_soft_moe.onnx"
gate_cpu = C.CorruptionClassifier(**moe.gate.config)
experts_cpu = [R.ConvAutoencoder(**e.config) for e in moe.experts]
moe_cpu = M.SoftMoE(gate_cpu, experts_cpu, float(moe.tau)).eval()
moe_cpu.load_state_dict({k: v.cpu() for k, v in moe.state_dict().items()})
export = M.MoEExport(moe_cpu).eval()
kw = dict(input_names=["input"], output_names=["output", "weights", "logits"], opset_version=17,
          dynamic_axes={n: {0: "batch"} for n in ("input", "output", "weights", "logits")})
with torch.no_grad():
    try:
        torch.onnx.export(export, torch.rand(1, 3, 128, 128), str(ONNX_PATH), dynamo=False, **kw)
    except TypeError:
        torch.onnx.export(export, torch.rand(1, 3, 128, 128), str(ONNX_PATH), **kw)
import onnx, onnxruntime as ort
onnx.checker.check_model(onnx.load(str(ONNX_PATH)))
sess = ort.InferenceSession(str(ONNX_PATH), providers=["CPUExecutionProvider"])
parity = {}
for split, xs in [("val", torch.cat([b[0] for _, b in zip(range(2), val_loader)])[:64]),
                  ("test", torch.cat([b[0] for _, b in zip(range(1), test_loader)])[:64])]:
    with torch.no_grad():
        ref = export(xs)
    o = sess.run(None, {"input": xs.numpy()})
    parity[split] = {"max_abs_diff_output": float(np.abs(o[0] - ref[0].numpy()).max()),
                     "max_abs_diff_weights": float(np.abs(o[1] - ref[1].numpy()).max()),
                     "argmax_agreement": float((o[1].argmax(1) == ref[1].argmax(1).numpy()).mean()), "n": len(xs)}
    parity[split]["passed"] = parity[split]["max_abs_diff_output"] < 1e-4 and parity[split]["max_abs_diff_weights"] < 1e-4
latency = R.onnx_latency_ms(ONNX_PATH, n_runs=100 if MODE == "full" else 20)
size_mb = ONNX_PATH.stat().st_size / 1e6
print(json.dumps({"parity": parity, "cpu_latency_batch1": latency, "size_mb": round(size_mb, 1)}, indent=1))
assert all(v["passed"] for v in parity.values()), "ONNX parity failed"
sidecar = {"task": "soft_moe_restoration", "onnx_file": ONNX_PATH.name, "opset": 17, "tau": float(moe.tau), "branches": BR,
           "outputs": {"output": "[batch,3,128,128] restored image in [0,1]", "weights": "[batch,4] routing weights w0..w3 (sum to 1)",
                       "logits": "[batch,4] gate logits G(x) (before /tau)"},
           "input": {"name": "input", "shape": ["batch", 3, 128, 128], "range": [0, 1], "layout": "NCHW RGB",
                     "preprocess": "PIL convert('RGB') -> resize 128x128 BICUBIC -> /255"},
           "hyperparameters": BEST_HP, "gate_config": moe.gate.config, "expert_configs": [e.config for e in moe.experts],
           "onnx_parity": parity, "cpu_latency_ms": latency}
pdt.save_json(sidecar, DIRS["models"] / "task3_soft_moe.json", indent=1)
if run:
    run.summary.update({"onnx/max_abs_diff_output": max(p["max_abs_diff_output"] for p in parity.values()),
                        "onnx/latency_ms": latency["median_ms"], "onnx/size_mb": size_mb})
    art = wandb.Artifact("task3-soft-moe-onnx", type="model", metadata=sidecar)
    art.add_file(str(ONNX_PATH)); art.add_file(str(DIRS["models"] / "task3_soft_moe.json")); run.log_artifact(art); run.finish()

## 10. Summary

In [ ]:
summary = {"mode": MODE, "data_source": data.source, "task2_sources": {str(k): str(v) for k, v in T2.items()},
           "best_trial": BEST_NUM, "best_hp": BEST_HP, "optuna_states": states,
           "initial_val_score": ev0["score"], "final_val_score": ck["val"]["score"], "best_step": ck["step"],
           "test_all_corrupted": by_type.loc["all corrupted"].dropna().round(4).to_dict(),
           "routing": {"usage": dict(zip(BR, usage.round(4).tolist())), "inactive": inactive,
                       "dominating_unrelated": [f"{b} on {c}" for b, c, _ in dominating],
                       "dominant_frac": float(dominant_frac), "distributed_frac": float(distributed_frac)},
           "compound": comp_tab[["hard_ssim", "soft_ssim", "soft better than hard (SSIM)"]].round(4).to_dict(),
           "onnx_parity_passed": all(v["passed"] for v in parity.values()), "unit_tests_passed": int(tests_df.passed.sum())}
pdt.save_json(summary, DIRS["results"] / "summary.json", indent=1)
print(json.dumps(summary, indent=1, default=str))
print("\nOutputs:")
for d in DIRS.values():
    for p in sorted(d.rglob("*")):
        if p.is_file():
            print(f"  {p.relative_to(WORK_DIR)}  ({p.stat().st_size / 1e6:.2f} MB)")
print("\nSMOKE TEST PASSED — switch MODE to 'full' and Save & Run All." if MODE == "smoke" else "\nTASK 3 COMPLETE.")